# T-Lab RecSys 2026: Causal GPT-2 Sequential Recommender System
### End-to-End Training, Hyperparameter Tuning & Evaluation on Kaggle (T4 / P100 GPU)

This notebook trains a **Causal GPT-2 Transformer from scratch** for Next-Item Prediction on the **Yambda** dataset (Yandex Music).

**Pre-flight Checklist on Kaggle:**
1. In the right sidebar under **Settings -> Accelerator**, select **GPU T4 x 2** or **GPU P100**.
2. In **Settings -> Internet**, toggle **Internet ON**.
3. Attach your Yambda dataset under `/kaggle/input/` (via "+ Add Input" in the top right), or allow auto-download from Hugging Face.

## 1. Verify GPU Accelerator

In [ ]:
!nvidia-smi

import os
# Prevent CUDA memory fragmentation on long-running sequence training
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Workspace & Codebase Setup (Self-Contained)
This cell automatically unpacks all updated production scripts (`data_utils.py`, `dataset.py`, `model.py`, `train_base.py`, and `sweep.py`) into the active directory. It works seamlessly whether cloned from Git or uploaded directly as a single `.ipynb`.

In [ ]:
import os
import sys
import base64

# Ensure CUDA expandable segments is enabled in current process
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Check if user cloned the repo into a subfolder
subdirs = [
    "/kaggle/working/music-recommender-system",
    "/kaggle/working/Music-Recommender-System",
]
for sd in subdirs:
    if os.path.exists(os.path.join(sd, "train_base.py")):
        print(f"Found cloned repo at {sd}, switching directory...")
        os.chdir(sd)
        break

# 2. Unpack embedded latest modules into current working directory
EMBEDDED_MODULES = {
  "data_utils.py": "...",
  "dataset.py": "IiIiCmRhdGFzZXQucHkKLS0tLS0tLS0tLQpQeVRvcmNoIERhdGFzZXQgYW5kIERhdGFMb2FkZXIgZm9yIEF1dG9yZWdyZXNzaXZlIFNlcXVlbnRpYWwgUmVjb21tZW5kZXIgU3lzdGVtcyAoTmV4dCBJdGVtIFByZWRpY3Rpb24pLgoKSW1wbGVtZW50czoKMS4gQ2hyb25vbG9naWNhbCBhZ2dyZWdhdGlvbiBwZXIgdXNlciAoc29ydGVkIGJ5IHRpbWVzdGFtcCkuCjIuIEZpeGVkLWxlbmd0aCBzZXF1ZW5jZSB3aW5kb3dpbmcgKG1heF9sZW5ndGgpLgozLiBDYXVzYWwgbGFuZ3VhZ2UgbW9kZWxpbmcgdGFyZ2V0IGFsaWdubWVudDoKICAgLSBJbnB1dCBTZXF1ZW5jZTogIFtzXzEsIHNfMiwgLi4uLCBzX2tdCiAgIC0gU2hpZnRlZCBMYWJlbDogICBbc18yLCBzXzMsIC4uLiwgc197aysxfV0KNC4gQXR0ZW50aW9uIG1hc2tpbmcgYW5kIHByb3BlciB6ZXJvLXBhZGRpbmcgKDxQQUQ+ID0gMCkuCjUuIExpc3RlbiBkdXJhdGlvbiBhbGlnbm1lbnQgW3JfMiwgcl8zLCAuLi4sIHJfe2srMX1dIHRvIHN1cHBvcnQgUGhhc2UgMiBSTCByZXdhcmQgZmluZS10dW5pbmcuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbCwgVHVwbGUsIFVuaW9uCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBEYXRhc2V0Cgpsb2dnaW5nLmJhc2ljQ29uZmlnKAogICAgbGV2ZWw9bG9nZ2luZy5JTkZPLAogICAgZm9ybWF0PSJbJShhc2N0aW1lKXNdIFslKGxldmVsbmFtZSlzXSAoJShuYW1lKXMpICUobWVzc2FnZSlzIiwKICAgIGRhdGVmbXQ9IiVZLSVtLSVkICVIOiVNOiVTIiwKKQpsb2dnZXIgPSBsb2dnaW5nLmdldExvZ2dlcigiRGF0YXNldCIpCgoKQGRhdGFjbGFzcwpjbGFzcyBSZWNCYXRjaDoKICAgICIiIgogICAgU3RydWN0dXJlZCBQeVRvcmNoIEJhdGNoIGZvciBDYXVzYWwgU2VxdWVudGlhbCBSZWNvbW1lbmRhdGlvbi4KICAgIAogICAgQXR0cmlidXRlczoKICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgaXRlbSB0b2tlbiBpbmRpY2VzLgogICAgICAgIGxhYmVsczogTG9uZ1RlbnNvciBbQiwgTF0gY29udGFpbmluZyBuZXh0LWl0ZW0gdGFyZ2V0IHRva2VuIGluZGljZXMuCiAgICAgICAgYXR0ZW50aW9uX21hc2s6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgMSBmb3IgdmFsaWQgaXRlbXMsIDAgZm9yIDxQQUQ+LgogICAgICAgIGR1cmF0aW9uczogRmxvYXRUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgbGlzdGVuaW5nIGR1cmF0aW9uIGluIHNlY29uZHMgKFJMIHJld2FyZHMpLgogICAgICAgIHVzZXJfaWRzOiBMaXN0W1VuaW9uW3N0ciwgaW50XV0gY29udGFpbmluZyByYXcgdXNlciBpZGVudGlmaWVycyBmb3IgZXZhbHVhdGlvbiBncm91cGluZy4KICAgICIiIgogICAgaW5wdXRfaWRzOiB0b3JjaC5UZW5zb3IKICAgIGxhYmVsczogdG9yY2guVGVuc29yCiAgICBhdHRlbnRpb25fbWFzazogdG9yY2guVGVuc29yCiAgICBkdXJhdGlvbnM6IHRvcmNoLlRlbnNvcgogICAgdXNlcl9pZHM6IExpc3RbVW5pb25bc3RyLCBpbnRdXQoKICAgIGRlZiB0byhzZWxmLCBkZXZpY2U6IFVuaW9uW3RvcmNoLmRldmljZSwgc3RyXSkgLT4gUmVjQmF0Y2g6CiAgICAgICAgIiIiVHJhbnNmZXJzIGJhdGNoIHRlbnNvcnMgdG8gdGFyZ2V0IGNvbXB1dGUgZGV2aWNlIChDUFUvR1BVL01QUykuIiIiCiAgICAgICAgcmV0dXJuIFJlY0JhdGNoKAogICAgICAgICAgICBpbnB1dF9pZHM9c2VsZi5pbnB1dF9pZHMudG8oZGV2aWNlKSwKICAgICAgICAgICAgbGFiZWxzPXNlbGYubGFiZWxzLnRvKGRldmljZSksCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPXNlbGYuYXR0ZW50aW9uX21hc2sudG8oZGV2aWNlKSwKICAgICAgICAgICAgZHVyYXRpb25zPXNlbGYuZHVyYXRpb25zLnRvKGRldmljZSksCiAgICAgICAgICAgIHVzZXJfaWRzPXNlbGYudXNlcl9pZHMsCiAgICAgICAgKQoKCmNsYXNzIFNlcXVlbnRpYWxSZWNEYXRhc2V0KERhdGFzZXQpOgogICAgIiIiCiAgICBQeVRvcmNoIERhdGFzZXQgZ2VuZXJhdGluZyBjYXVzYWwgKGlucHV0LCBsYWJlbCkgc2VxdWVuY2UgcGFpcnMgZm9yIE5leHQgSXRlbSBQcmVkaWN0aW9uLgoKICAgIEFyZ3M6CiAgICAgICAgZGY6IERhdGFGcmFtZSBjb250YWluaW5nIGludGVyYWN0aW9uIGxvZ3MuIE11c3QgY29udGFpbiB1c2VyX2NvbCwgaXRlbV90b2tlbl9jb2wsCiAgICAgICAgICAgIHRpbWVzdGFtcF9jb2wsIGFuZCBvcHRpb25hbGx5IGR1cmF0aW9uX2NvbC4KICAgICAgICBtYXhfbGVuZ3RoOiBNYXhpbXVtIHNlcXVlbmNlIGxlbmd0aCAoY29udGV4dCB3aW5kb3cpIGZvciB0aGUgR1BULTIgbW9kZWwuCiAgICAgICAgdXNlcl9jb2w6IENvbHVtbiBuYW1lIGZvciB1c2VyIGlkZW50aWZpZXJzLgogICAgICAgIGl0ZW1fdG9rZW5fY29sOiBDb2x1bW4gbmFtZSBmb3IgbWFwcGVkIGNvbnRpbnVvdXMgaXRlbSB0b2tlbnMgKDEgLi4uIE4pLgogICAgICAgIHRpbWVzdGFtcF9jb2w6IENvbHVtbiBuYW1lIGZvciBVTklYIHRpbWVzdGFtcHMuCiAgICAgICAgZHVyYXRpb25fY29sOiBDb2x1bW4gbmFtZSBmb3IgbGlzdGVuIGR1cmF0aW9uIGluIHNlY29uZHMuCiAgICAgICAgc2xpZGluZ193aW5kb3c6IElmIFRydWUsIHVzZXMgc2xpZGluZyB3aW5kb3cgY2h1bmtpbmcgdG8gZXh0cmFjdCBtdWx0aXBsZSBzdWJzZXF1ZW5jZXMKICAgICAgICAgICAgZnJvbSBsb25nIHVzZXIgaGlzdG9yaWVzIChyZWNvbW1lbmRlZCBmb3IgdHJhaW5pbmcgZGF0YSBhdWdtZW50YXRpb24pLgogICAgICAgICAgICBJZiBGYWxzZSwgcmV0YWlucyBvbmx5IHRoZSBtb3N0IHJlY2VudCBzZXF1ZW5jZSAocmVjb21tZW5kZWQgZm9yIHZhbGlkYXRpb24vdGVzdCBldmFsdWF0aW9uKS4KICAgICAgICBzdHJpZGU6IFN0cmlkZSBzdGVwIHNpemUgd2hlbiBzbGlkaW5nX3dpbmRvdyBpcyBUcnVlLiBEZWZhdWx0cyB0byBtYXhfbGVuZ3RoIC8vIDIuCiAgICAgICAgcGFkX3Rva2VuX2lkOiBTcGVjaWFsIHRva2VuIGluZGV4IGZvciBwYWRkaW5nIChkZWZhdWx0IDApLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICBoaXN0b3J5X2RmOiBPcHRpb25hbFtwZC5EYXRhRnJhbWVdID0gTm9uZSwKICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgICAgICB1c2VyX2NvbDogc3RyID0gInVpZCIsCiAgICAgICAgaXRlbV90b2tlbl9jb2w6IHN0ciA9ICJpdGVtX3Rva2VuIiwKICAgICAgICB0aW1lc3RhbXBfY29sOiBzdHIgPSAidGltZXN0YW1wIiwKICAgICAgICBkdXJhdGlvbl9jb2w6IHN0ciA9ICJsaXN0ZW5fZHVyYXRpb25fc2VjIiwKICAgICAgICBzbGlkaW5nX3dpbmRvdzogYm9vbCA9IEZhbHNlLAogICAgICAgIHN0cmlkZTogT3B0aW9uYWxbaW50XSA9IE5vbmUsCiAgICAgICAgcGFkX3Rva2VuX2lkOiBpbnQgPSAwLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubWF4X2xlbmd0aDogaW50ID0gbWF4X2xlbmd0aAogICAgICAgIHNlbGYucGFkX3Rva2VuX2lkOiBpbnQgPSBwYWRfdG9rZW5faWQKICAgICAgICBzZWxmLnNsaWRpbmdfd2luZG93OiBib29sID0gc2xpZGluZ193aW5kb3cKICAgICAgICBzZWxmLnN0cmlkZTogaW50ID0gc3RyaWRlIGlmIHN0cmlkZSBpcyBub3QgTm9uZSBlbHNlIG1heCgxLCBtYXhfbGVuZ3RoIC8vIDIpCgogICAgICAgICMgUmVzb2x2ZSB1c2VyX2NvbCBhbGlhcyBpZiBuZWVkZWQKICAgICAgICBpZiB1c2VyX2NvbCBub3QgaW4gZGYuY29sdW1uczoKICAgICAgICAgICAgaWYgInVpZCIgaW4gZGYuY29sdW1uczoKICAgICAgICAgICAgICAgIHVzZXJfY29sID0gInVpZCIKICAgICAgICAgICAgZWxpZiAidXNlcl9pZCIgaW4gZGYuY29sdW1uczoKICAgICAgICAgICAgICAgIHVzZXJfY29sID0gInVzZXJfaWQiCgogICAgICAgICMgSW4gZXZhbHVhdGlvbiAodmFsL3Rlc3QpLCBjb25jYXRlbmF0ZSBoaXN0b3JpY2FsIGludGVyYWN0aW9ucyB3aXRoIHRhcmdldCBzcGxpdAogICAgICAgICMgdG8gZW5zdXJlIHVzZXJzIGhhdmUgZnVsbCBjaHJvbm9sb2dpY2FsIGNvbnRleHQgdXAgdG8gdGhlIGV2YWx1YXRpb24gdGFyZ2V0LgogICAgICAgIGlmIGhpc3RvcnlfZGYgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICMgRmlsdGVyIGhpc3RvcnkgdG8gb25seSB1c2VycyBwcmVzZW50IGluIGN1cnJlbnQgdGFyZ2V0IHNwbGl0CiAgICAgICAgICAgIGFjdGl2ZV91c2VycyA9IHNldChkZlt1c2VyX2NvbF0udW5pcXVlKCkpCiAgICAgICAgICAgIGZpbHRlcmVkX2hpc3RvcnkgPSBoaXN0b3J5X2RmW2hpc3RvcnlfZGZbdXNlcl9jb2xdLmlzaW4oYWN0aXZlX3VzZXJzKV0KICAgICAgICAgICAgY29tYmluZWRfZGYgPSBwZC5jb25jYXQoW2ZpbHRlcmVkX2hpc3RvcnksIGRmXSwgaWdub3JlX2luZGV4PVRydWUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY29tYmluZWRfZGYgPSBkZgoKICAgICAgICAjIFN0b3JlIHNlcXVlbmNlIHNhbXBsZXM6IExpc3Qgb2YgdHVwbGVzICh1c2VyX2lkLCBpdGVtX3NlcSwgZHVyYXRpb25fc2VxKQogICAgICAgIHNlbGYuc2FtcGxlczogTGlzdFtUdXBsZVtVbmlvbltzdHIsIGludF0sIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldXSA9IFtdCiAgICAgICAgc2VsZi5fYnVpbGRfc2VxdWVuY2VzKAogICAgICAgICAgICBkZj1jb21iaW5lZF9kZiwKICAgICAgICAgICAgdXNlcl9jb2w9dXNlcl9jb2wsCiAgICAgICAgICAgIGl0ZW1fdG9rZW5fY29sPWl0ZW1fdG9rZW5fY29sLAogICAgICAgICAgICB0aW1lc3RhbXBfY29sPXRpbWVzdGFtcF9jb2wsCiAgICAgICAgICAgIGR1cmF0aW9uX2NvbD1kdXJhdGlvbl9jb2wsCiAgICAgICAgKQoKICAgIGRlZiBfYnVpbGRfc2VxdWVuY2VzKAogICAgICAgIHNlbGYsCiAgICAgICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICB1c2VyX2NvbDogc3RyLAogICAgICAgIGl0ZW1fdG9rZW5fY29sOiBzdHIsCiAgICAgICAgdGltZXN0YW1wX2NvbDogc3RyLAogICAgICAgIGR1cmF0aW9uX2NvbDogc3RyLAogICAgKSAtPiBOb25lOgogICAgICAgICIiIkdyb3VwcyB1c2VyIGludGVyYWN0aW9ucywgc29ydHMgY2hyb25vbG9naWNhbGx5LCBhbmQgY29uc3RydWN0cyBzZXF1ZW5jZXMuIiIiCiAgICAgICAgIyBFbnN1cmUgY2hyb25vbG9naWNhbCBvcmRlcmluZwogICAgICAgIHNvcnRlZF9kZiA9IGRmLnNvcnRfdmFsdWVzKGJ5PVt1c2VyX2NvbCwgdGltZXN0YW1wX2NvbF0pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICAgICBoYXNfZHVyYXRpb24gPSBkdXJhdGlvbl9jb2wgaW4gc29ydGVkX2RmLmNvbHVtbnMKCiAgICAgICAgZ3JvdXBlZCA9IHNvcnRlZF9kZi5ncm91cGJ5KHVzZXJfY29sKQogICAgICAgIHNraXBwZWRfc2hvcnQgPSAwCgogICAgICAgIGZvciB1c2VyX2lkLCBncm91cCBpbiBncm91cGVkOgogICAgICAgICAgICBpdGVtX3NlcSA9IGdyb3VwW2l0ZW1fdG9rZW5fY29sXS52YWx1ZXMuYXN0eXBlKG5wLmludDY0KQogICAgICAgICAgICBkdXJhdGlvbl9zZXEgPSAoCiAgICAgICAgICAgICAgICBncm91cFtkdXJhdGlvbl9jb2xdLnZhbHVlcy5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICAgICAgICAgIGlmIGhhc19kdXJhdGlvbgogICAgICAgICAgICAgICAgZWxzZSBucC5vbmVzKGxlbihpdGVtX3NlcSksIGR0eXBlPW5wLmZsb2F0MzIpICogMzAuMAogICAgICAgICAgICApCgogICAgICAgICAgICAjIENhdXNhbCBMTSBuZXh0LWl0ZW0gcHJlZGljdGlvbiByZXF1aXJlcyBhdCBsZWFzdCAyIHRva2VucyAoMSBpbnB1dCwgMSBsYWJlbCkKICAgICAgICAgICAgaWYgbGVuKGl0ZW1fc2VxKSA8IDI6CiAgICAgICAgICAgICAgICBza2lwcGVkX3Nob3J0ICs9IDEKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBpZiBzZWxmLnNsaWRpbmdfd2luZG93OgogICAgICAgICAgICAgICAgIyBTbGlkaW5nIHdpbmRvdyBjaHVua2luZyBmb3IgdHJhaW5pbmcgZGF0YSBhdWdtZW50YXRpb24KICAgICAgICAgICAgICAgICMgQSB3aW5kb3cgb2Ygc2l6ZSAobWF4X2xlbmd0aCArIDEpIHByb3ZpZGVzIG1heF9sZW5ndGggaW5wdXRzIGFuZCBtYXhfbGVuZ3RoIHNoaWZ0ZWQgbGFiZWxzCiAgICAgICAgICAgICAgICB3aW5kb3dfc2l6ZSA9IHNlbGYubWF4X2xlbmd0aCArIDEKICAgICAgICAgICAgICAgIGlmIGxlbihpdGVtX3NlcSkgPD0gd2luZG93X3NpemU6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5zYW1wbGVzLmFwcGVuZCgodXNlcl9pZCwgaXRlbV9zZXEsIGR1cmF0aW9uX3NlcSkpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIGZvciBzdGFydF9pZHggaW4gcmFuZ2UoMCwgbGVuKGl0ZW1fc2VxKSAtIDEsIHNlbGYuc3RyaWRlKToKICAgICAgICAgICAgICAgICAgICAgICAgZW5kX2lkeCA9IG1pbihzdGFydF9pZHggKyB3aW5kb3dfc2l6ZSwgbGVuKGl0ZW1fc2VxKSkKICAgICAgICAgICAgICAgICAgICAgICAgY2h1bmtfaXRlbXMgPSBpdGVtX3NlcVtzdGFydF9pZHg6ZW5kX2lkeF0KICAgICAgICAgICAgICAgICAgICAgICAgY2h1bmtfZHVyYXRpb25zID0gZHVyYXRpb25fc2VxW3N0YXJ0X2lkeDplbmRfaWR4XQogICAgICAgICAgICAgICAgICAgICAgICBpZiBsZW4oY2h1bmtfaXRlbXMpID49IDI6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLnNhbXBsZXMuYXBwZW5kKCh1c2VyX2lkLCBjaHVua19pdGVtcywgY2h1bmtfZHVyYXRpb25zKSkKICAgICAgICAgICAgICAgICAgICAgICAgaWYgZW5kX2lkeCA9PSBsZW4oaXRlbV9zZXEpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICMgRXZhbHVhdGlvbiBtb2RlOiByZXRhaW4gdGhlIG1vc3QgcmVjZW50IGhpc3RvcnkgdXAgdG8gKG1heF9sZW5ndGggKyAxKQogICAgICAgICAgICAgICAgY2h1bmtfaXRlbXMgPSBpdGVtX3NlcVstKHNlbGYubWF4X2xlbmd0aCArIDEpOl0KICAgICAgICAgICAgICAgIGNodW5rX2R1cmF0aW9ucyA9IGR1cmF0aW9uX3NlcVstKHNlbGYubWF4X2xlbmd0aCArIDEpOl0KICAgICAgICAgICAgICAgIHNlbGYuc2FtcGxlcy5hcHBlbmQoKHVzZXJfaWQsIGNodW5rX2l0ZW1zLCBjaHVua19kdXJhdGlvbnMpKQoKICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgIkNvbnN0cnVjdGVkICVkIHNlcXVlbmNlIHNhbXBsZXMgZnJvbSAlZCB1c2VycyAoc2tpcHBlZCAlZCB1c2VycyB3aXRoIDwyIGludGVyYWN0aW9ucykuICIKICAgICAgICAgICAgInNsaWRpbmdfd2luZG93PSVzLCBtYXhfbGVuZ3RoPSVkIiwKICAgICAgICAgICAgbGVuKHNlbGYuc2FtcGxlcyksCiAgICAgICAgICAgIGxlbihncm91cGVkKSwKICAgICAgICAgICAgc2tpcHBlZF9zaG9ydCwKICAgICAgICAgICAgc2VsZi5zbGlkaW5nX3dpbmRvdywKICAgICAgICAgICAgc2VsZi5tYXhfbGVuZ3RoLAogICAgICAgICkKCiAgICBkZWYgX19sZW5fXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLnNhbXBsZXMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeDogaW50KSAtPiBEaWN0W3N0ciwgVW5pb25bbnAubmRhcnJheSwgVW5pb25bc3RyLCBpbnRdXV06CiAgICAgICAgIiIiCiAgICAgICAgUmV0cmlldmVzIGEgc2luZ2xlIHNlcXVlbmNlIHNhbXBsZSwgYWxpZ25zIGlucHV0cyBhbmQgc2hpZnRlZCBjYXVzYWwgbGFiZWxzLAogICAgICAgIGFuZCBhcHBsaWVzIHplcm8tcGFkZGluZyB0byByZWFjaCBtYXhfbGVuZ3RoLgoKICAgICAgICBMZXQgcmF3IGl0ZW1zIGJlIFtzXzEsIHNfMiwgLi4uLCBzX2ssIHNfe2srMX1dLgogICAgICAgIC0gSW5wdXQgc2VxdWVuY2U6ICAgW3NfMSwgc18yLCAuLi4sIHNfa10KICAgICAgICAtIFNoaWZ0ZWQgTGFiZWw6ICAgIFtzXzIsIHNfMywgLi4uLCBzX3trKzF9XQogICAgICAgIC0gU2hpZnRlZCBEdXJhdGlvbjogW3JfMiwgcl8zLCAuLi4sIHJfe2srMX1dCiAgICAgICAgIiIiCiAgICAgICAgdXNlcl9pZCwgaXRlbXMsIGR1cmF0aW9ucyA9IHNlbGYuc2FtcGxlc1tpZHhdCgogICAgICAgICMgU2hpZnQgc2VxdWVuY2UgZm9yIG5leHQtdG9rZW4gcHJlZGljdGlvbgogICAgICAgIHJhd19pbnB1dHMgPSBpdGVtc1s6LTFdCiAgICAgICAgcmF3X2xhYmVscyA9IGl0ZW1zWzE6XQogICAgICAgIHJhd19kdXJhdGlvbnMgPSBkdXJhdGlvbnNbMTpdCgogICAgICAgICMgVHJ1bmNhdGUgaWYgbG9uZ2VyIHRoYW4gbWF4X2xlbmd0aCAoZS5nLiBmcm9tIHJpZ2h0bW9zdCByZWNlbnQgaGlzdG9yeSkKICAgICAgICBpZiBsZW4ocmF3X2lucHV0cykgPiBzZWxmLm1heF9sZW5ndGg6CiAgICAgICAgICAgIHJhd19pbnB1dHMgPSByYXdfaW5wdXRzWy1zZWxmLm1heF9sZW5ndGg6XQogICAgICAgICAgICByYXdfbGFiZWxzID0gcmF3X2xhYmVsc1stc2VsZi5tYXhfbGVuZ3RoOl0KICAgICAgICAgICAgcmF3X2R1cmF0aW9ucyA9IHJhd19kdXJhdGlvbnNbLXNlbGYubWF4X2xlbmd0aDpdCgogICAgICAgIGN1cnJfbGVuID0gbGVuKHJhd19pbnB1dHMpCiAgICAgICAgcGFkX2xlbiA9IHNlbGYubWF4X2xlbmd0aCAtIGN1cnJfbGVuCgogICAgICAgICMgUmlnaHQtcGFkZGluZyB3aXRoIHBhZF90b2tlbl9pZCAoMCkKICAgICAgICAjIElucHV0IElEczogW3Rva2VuXzEsIHRva2VuXzIsIC4uLiwgdG9rZW5faywgMCwgLi4uLCAwXQogICAgICAgIGlucHV0X2lkcyA9IG5wLnBhZChyYXdfaW5wdXRzLCAoMCwgcGFkX2xlbiksIGNvbnN0YW50X3ZhbHVlcz1zZWxmLnBhZF90b2tlbl9pZCkKICAgICAgICAjIExhYmVsczogW3Rva2VuXzIsIHRva2VuXzMsIC4uLiwgdG9rZW5fe2srMX0sIDAsIC4uLiwgMF0KICAgICAgICBsYWJlbHMgPSBucC5wYWQocmF3X2xhYmVscywgKDAsIHBhZF9sZW4pLCBjb25zdGFudF92YWx1ZXM9c2VsZi5wYWRfdG9rZW5faWQpCiAgICAgICAgIyBBdHRlbnRpb24gTWFzazogMSBmb3IgYWN0aXZlIGl0ZW0gdG9rZW5zLCAwIGZvciA8UEFEPgogICAgICAgIGF0dGVudGlvbl9tYXNrID0gbnAuemVyb3Moc2VsZi5tYXhfbGVuZ3RoLCBkdHlwZT1ucC5pbnQ2NCkKICAgICAgICBhdHRlbnRpb25fbWFza1s6Y3Vycl9sZW5dID0gMQogICAgICAgICMgRHVyYXRpb25zOiBhbGlnbmVkIHdpdGggbGFiZWxzIGZvciBSTCByZXdhcmRzCiAgICAgICAgZHVyYXRpb25zX3BhZGRlZCA9IG5wLnBhZChyYXdfZHVyYXRpb25zLCAoMCwgcGFkX2xlbiksIGNvbnN0YW50X3ZhbHVlcz0wLjApCgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJpbnB1dF9pZHMiOiBpbnB1dF9pZHMsCiAgICAgICAgICAgICJsYWJlbHMiOiBsYWJlbHMsCiAgICAgICAgICAgICJhdHRlbnRpb25fbWFzayI6IGF0dGVudGlvbl9tYXNrLAogICAgICAgICAgICAiZHVyYXRpb25zIjogZHVyYXRpb25zX3BhZGRlZCwKICAgICAgICAgICAgInVzZXJfaWQiOiB1c2VyX2lkLAogICAgICAgIH0KCgpkZWYgcmVjc3lzX2NvbGxhdGVfZm4oYmF0Y2hfbGlzdDogTGlzdFtEaWN0W3N0ciwgVW5pb25bbnAubmRhcnJheSwgVW5pb25bc3RyLCBpbnRdXV1dKSAtPiBSZWNCYXRjaDoKICAgICIiIgogICAgQ29sbGF0ZSBmdW5jdGlvbiB0aGF0IGJhdGNoZXMgaW5kaXZpZHVhbCBkaWN0aW9uYXJ5IHNhbXBsZXMgaW50byBhIHVuaWZpZWQgUmVjQmF0Y2guCiAgICAiIiIKICAgIGlucHV0X2lkcyA9IHRvcmNoLmZyb21fbnVtcHkobnAuc3RhY2soW2JbImlucHV0X2lkcyJdIGZvciBiIGluIGJhdGNoX2xpc3RdKSkubG9uZygpCiAgICBsYWJlbHMgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFtiWyJsYWJlbHMiXSBmb3IgYiBpbiBiYXRjaF9saXN0XSkpLmxvbmcoKQogICAgYXR0ZW50aW9uX21hc2sgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFtiWyJhdHRlbnRpb25fbWFzayJdIGZvciBiIGluIGJhdGNoX2xpc3RdKSkubG9uZygpCiAgICBkdXJhdGlvbnMgPSB0b3JjaC5mcm9tX251bXB5KG5wLnN0YWNrKFtiWyJkdXJhdGlvbnMiXSBmb3IgYiBpbiBiYXRjaF9saXN0XSkpLmZsb2F0KCkKICAgIHVzZXJfaWRzID0gW2JbInVzZXJfaWQiXSBmb3IgYiBpbiBiYXRjaF9saXN0XQoKICAgIHJldHVybiBSZWNCYXRjaCgKICAgICAgICBpbnB1dF9pZHM9aW5wdXRfaWRzLAogICAgICAgIGxhYmVscz1sYWJlbHMsCiAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2ssCiAgICAgICAgZHVyYXRpb25zPWR1cmF0aW9ucywKICAgICAgICB1c2VyX2lkcz11c2VyX2lkcywKICAgICkKCgpkZWYgY3JlYXRlX2RhdGFsb2FkZXIoCiAgICBkZjogcGQuRGF0YUZyYW1lLAogICAgaGlzdG9yeV9kZjogT3B0aW9uYWxbcGQuRGF0YUZyYW1lXSA9IE5vbmUsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgIGJhdGNoX3NpemU6IGludCA9IDEyOCwKICAgIHNodWZmbGU6IGJvb2wgPSBUcnVlLAogICAgc2xpZGluZ193aW5kb3c6IGJvb2wgPSBGYWxzZSwKICAgIHN0cmlkZTogT3B0aW9uYWxbaW50XSA9IE5vbmUsCiAgICB1c2VyX2NvbDogc3RyID0gInVpZCIsCiAgICBudW1fd29ya2VyczogaW50ID0gMCwKICAgIHBpbl9tZW1vcnk6IGJvb2wgPSBGYWxzZSwKICAgIHBhZF90b2tlbl9pZDogaW50ID0gMCwKKSAtPiBEYXRhTG9hZGVyOgogICAgIiIiCiAgICBGYWN0b3J5IGZ1bmN0aW9uIGNyZWF0aW5nIGFuIG9wdGltaXplZCBQeVRvcmNoIERhdGFMb2FkZXIgZm9yIHNlcXVlbnRpYWwgcmVjb21tZW5kYXRpb24uCiAgICAiIiIKICAgIGRhdGFzZXQgPSBTZXF1ZW50aWFsUmVjRGF0YXNldCgKICAgICAgICBkZj1kZiwKICAgICAgICBoaXN0b3J5X2RmPWhpc3RvcnlfZGYsCiAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLAogICAgICAgIHVzZXJfY29sPXVzZXJfY29sLAogICAgICAgIHNsaWRpbmdfd2luZG93PXNsaWRpbmdfd2luZG93LAogICAgICAgIHN0cmlkZT1zdHJpZGUsCiAgICAgICAgcGFkX3Rva2VuX2lkPXBhZF90b2tlbl9pZCwKICAgICkKCiAgICByZXR1cm4gRGF0YUxvYWRlcigKICAgICAgICBkYXRhc2V0PWRhdGFzZXQsCiAgICAgICAgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9c2h1ZmZsZSwKICAgICAgICBudW1fd29ya2Vycz1udW1fd29ya2VycywKICAgICAgICBjb2xsYXRlX2ZuPXJlY3N5c19jb2xsYXRlX2ZuLAogICAgICAgIHBpbl9tZW1vcnk9cGluX21lbW9yeSwKICAgICAgICBkcm9wX2xhc3Q9RmFsc2UsCiAgICApCg==",
  "model.py": "IiIiCm1vZGVsLnB5Ci0tLS0tLS0tCkNhdXNhbCBMYW5ndWFnZSBNb2RlbCAoR1BULTIpIGZvciBBdXRvcmVncmVzc2l2ZSBOZXh0LUl0ZW0gUmVjb21tZW5kYXRpb24uCgpJbml0aWFsaXplcyBhIGNvbXBhY3QgR1BULTIgYXJjaGl0ZWN0dXJlIGZyb20gc2NyYXRjaCAobm8gcHJlLXRyYWluZWQgTkxQIHdlaWdodHMpCnRhaWxvcmVkIGZvciBpdGVtIGNhdGFsb2dzIGFuZCBzZXF1ZW50aWFsIGludGVyYWN0aW9uIGR5bmFtaWNzLgoKQXJjaGl0ZWN0dXJlIFNwZWNpZmljYXRpb25zOgogICAgLSBCYXNlIENsYXNzOiB0cmFuc2Zvcm1lcnMuR1BUMkxNSGVhZE1vZGVsIChpbml0aWFsaXplZCBmcm9tIHNjcmF0Y2gpCiAgICAtIHZvY2FiX3NpemUgPSBudW1faXRlbXMgKyAxIChJbmRleCAwIGlzIDxQQUQ+KQogICAgLSBuX2VtYmQgPSAxMjggKENvbXBhY3QgbGF0ZW50IGVtYmVkZGluZ3MpCiAgICAtIG5fbGF5ZXIgPSAzIChUcmFuc2Zvcm1lciBkZWNvZGVyIGJsb2NrcykKICAgIC0gbl9oZWFkID0gNCAoTXVsdGktaGVhZCBzZWxmLWF0dGVudGlvbiBoZWFkcykKICAgIC0gbl9wb3NpdGlvbnMgPSBtYXhfbGVuZ3RoIChDb250ZXh0IHdpbmRvdyBsZW5ndGgpCiAgICAtIHBhZF90b2tlbl9pZCA9IDAKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIE9wdGlvbmFsLCBUdXBsZSwgVW5pb24KCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEdQVDJDb25maWcsIEdQVDJMTUhlYWRNb2RlbAoKbG9nZ2luZy5iYXNpY0NvbmZpZygKICAgIGxldmVsPWxvZ2dpbmcuSU5GTywKICAgIGZvcm1hdD0iWyUoYXNjdGltZSlzXSBbJShsZXZlbG5hbWUpc10gKCUobmFtZSlzKSAlKG1lc3NhZ2UpcyIsCiAgICBkYXRlZm10PSIlWS0lbS0lZCAlSDolTTolUyIsCikKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoIk1vZGVsIikKCgpAZGF0YWNsYXNzCmNsYXNzIFJlY1N5c091dHB1dDoKICAgICIiIgogICAgU3RhbmRhcmRpemVkIG1vZGVsIG91dHB1dCBjb250YWluZXIgZm9yIHNlcXVlbnRpYWwgcmVjb21tZW5kYXRpb24uCiAgICAKICAgIEF0dHJpYnV0ZXM6CiAgICAgICAgbG9naXRzOiBGbG9hdFRlbnNvciBbQiwgTCwgdm9jYWJfc2l6ZV0gdW5ub3JtYWxpemVkIHByZWRpY3Rpb24gc2NvcmVzLgogICAgICAgIGxvc3M6IE9wdGlvbmFsIHNjYWxhciBjcm9zcy1lbnRyb3B5IGxvc3MgKGlmIGxhYmVscyB3ZXJlIHByb3ZpZGVkKS4KICAgICAgICBsYXN0X2hpZGRlbl9zdGF0ZTogT3B0aW9uYWwgRmxvYXRUZW5zb3IgW0IsIEwsIG5fZW1iZF0gY29udGV4dHVhbCByZXByZXNlbnRhdGlvbnMuCiAgICAiIiIKICAgIGxvZ2l0czogdG9yY2guVGVuc29yCiAgICBsb3NzOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZQogICAgbGFzdF9oaWRkZW5fc3RhdGU6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lCgoKY2xhc3MgUmVjU3lzR1BUMihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBTZXF1ZW5jZS1iYXNlZCBSZWNvbW1lbmRhdGlvbiBNb2RlbCB1c2luZyBhIENhdXNhbCBHUFQtMiBUcmFuc2Zvcm1lciBpbml0aWFsaXplZCBmcm9tIHNjcmF0Y2guCgogICAgSW4gTmV4dC1JdGVtIFByZWRpY3Rpb24sIGVhY2ggaW50ZXJhY3Rpb24gdG9rZW4gc2VxdWVuY2UgW3NfMSwgLi4uLCBzX2tdIGlzIHByb2plY3RlZAogICAgaW50byBhbiBlbWJlZGRpbmcgc3BhY2UsIHBhc3NlZCB0aHJvdWdoIGNhdXNhbCBzZWxmLWF0dGVudGlvbiBsYXllcnMgd2l0aCBhdXRvcmVncmVzc2l2ZQogICAgbWFza2luZywgYW5kIHByb2plY3RlZCB0byBjYXRhbG9nIGxvZ2l0cyBbQiwgTCwgdm9jYWJfc2l6ZV0uCgogICAgQXJnczoKICAgICAgICBudW1faXRlbXM6IFRvdGFsIG51bWJlciBvZiBhY3RpdmUgaXRlbXMgaW4gY2F0YWxvZyAoZXhjbHVkaW5nIDxQQUQ+KS4KICAgICAgICBtYXhfbGVuZ3RoOiBNYXhpbXVtIHNlcXVlbmNlIGNvbnRleHQgbGVuZ3RoIChuX3Bvc2l0aW9ucykuCiAgICAgICAgbl9lbWJkOiBMYXRlbnQgZW1iZWRkaW5nIGRpbWVuc2lvbiAoZGVmYXVsdDogMTI4KS4KICAgICAgICBuX2xheWVyOiBOdW1iZXIgb2YgdHJhbnNmb3JtZXIgZGVjb2RlciBsYXllcnMgKGRlZmF1bHQ6IDMpLgogICAgICAgIG5faGVhZDogTnVtYmVyIG9mIGF0dGVudGlvbiBoZWFkcyAoZGVmYXVsdDogNCkuCiAgICAgICAgZHJvcG91dDogRHJvcG91dCBwcm9iYWJpbGl0eSBhcHBsaWVkIHRvIGVtYmVkZGluZ3MgYW5kIHJlc2lkdWFsIGNvbm5lY3Rpb25zLgogICAgICAgIHBhZF90b2tlbl9pZDogUmVzZXJ2ZWQgcGFkZGluZyB0b2tlbiBpbmRleCAoZGVmYXVsdDogMCkuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBudW1faXRlbXM6IGludCwKICAgICAgICBtYXhfbGVuZ3RoOiBpbnQgPSA2NCwKICAgICAgICBuX2VtYmQ6IGludCA9IDEyOCwKICAgICAgICBuX2xheWVyOiBpbnQgPSAzLAogICAgICAgIG5faGVhZDogaW50ID0gNCwKICAgICAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMSwKICAgICAgICBwYWRfdG9rZW5faWQ6IGludCA9IDAsCiAgICApIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5udW1faXRlbXM6IGludCA9IG51bV9pdGVtcwogICAgICAgIHNlbGYubWF4X2xlbmd0aDogaW50ID0gbWF4X2xlbmd0aAogICAgICAgIHNlbGYucGFkX3Rva2VuX2lkOiBpbnQgPSBwYWRfdG9rZW5faWQKCiAgICAgICAgIyBWb2NhYiBzaXplID0gbnVtX2l0ZW1zICsgMSAoaXRlbSBJRHMgMS4uTiwgcGx1cyAwIGZvciBQQUQpCiAgICAgICAgdm9jYWJfc2l6ZSA9IG51bV9pdGVtcyArIDEKCiAgICAgICAgIyBDb25maWd1cmUgY3VzdG9tIEdQVC0yIGFyY2hpdGVjdHVyZQogICAgICAgIHNlbGYuY29uZmlnID0gR1BUMkNvbmZpZygKICAgICAgICAgICAgdm9jYWJfc2l6ZT12b2NhYl9zaXplLAogICAgICAgICAgICBuX3Bvc2l0aW9ucz1tYXhfbGVuZ3RoLAogICAgICAgICAgICBuX2N0eD1tYXhfbGVuZ3RoLAogICAgICAgICAgICBuX2VtYmQ9bl9lbWJkLAogICAgICAgICAgICBuX2xheWVyPW5fbGF5ZXIsCiAgICAgICAgICAgIG5faGVhZD1uX2hlYWQsCiAgICAgICAgICAgIHJlc2lkX3Bkcm9wPWRyb3BvdXQsCiAgICAgICAgICAgIGVtYmRfcGRyb3A9ZHJvcG91dCwKICAgICAgICAgICAgYXR0bl9wZHJvcD1kcm9wb3V0LAogICAgICAgICAgICBwYWRfdG9rZW5faWQ9cGFkX3Rva2VuX2lkLAogICAgICAgICAgICBib3NfdG9rZW5faWQ9Tm9uZSwgICMgQ2xlYXIgZGVmYXVsdCB0ZXh0IE5MUCB0b2tlbnMKICAgICAgICAgICAgZW9zX3Rva2VuX2lkPU5vbmUsCiAgICAgICAgKQoKICAgICAgICAjIFNUUklDVCBSRVFVSVJFTUVOVDogSW5pdGlhbGl6ZWQgZnJvbSBzY3JhdGNoIChOT1QgZnJvbSBwcmUtdHJhaW5lZCB3ZWlnaHRzKQogICAgICAgIHNlbGYudHJhbnNmb3JtZXIgPSBHUFQyTE1IZWFkTW9kZWwoc2VsZi5jb25maWcpCgogICAgICAgICMgQ3Jvc3MtRW50cm9weSBMb3NzIHdpdGggaWdub3JlX2luZGV4PTAgKDxQQUQ+KQogICAgICAgIHNlbGYubG9zc19mY3QgPSBubi5Dcm9zc0VudHJvcHlMb3NzKGlnbm9yZV9pbmRleD1zZWxmLnBhZF90b2tlbl9pZCkKCiAgICAgICAgIyBDb3VudCBwYXJhbWV0ZXJzCiAgICAgICAgbnVtX3BhcmFtcyA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkpCiAgICAgICAgbnVtX3RyYWluYWJsZSA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKQogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICAiSW5pdGlhbGl6ZWQgUmVjU3lzR1BUMiBmcm9tIHNjcmF0Y2g6IHZvY2FiX3NpemU9JWQgKG51bV9pdGVtcz0lZCArIFBBRCksICIKICAgICAgICAgICAgIm5fZW1iZD0lZCwgbl9sYXllcj0lZCwgbl9oZWFkPSVkLCBtYXhfbGVuZ3RoPSVkLiAiCiAgICAgICAgICAgICJUb3RhbCBwYXJhbWV0ZXJzOiAlZCAodHJhaW5hYmxlOiAlZCkiLAogICAgICAgICAgICB2b2NhYl9zaXplLAogICAgICAgICAgICBudW1faXRlbXMsCiAgICAgICAgICAgIG5fZW1iZCwKICAgICAgICAgICAgbl9sYXllciwKICAgICAgICAgICAgbl9oZWFkLAogICAgICAgICAgICBtYXhfbGVuZ3RoLAogICAgICAgICAgICBudW1fcGFyYW1zLAogICAgICAgICAgICBudW1fdHJhaW5hYmxlLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZCgKICAgICAgICBzZWxmLAogICAgICAgIGlucHV0X2lkczogdG9yY2guVGVuc29yLAogICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbFt0b3JjaC5UZW5zb3JdID0gTm9uZSwKICAgICAgICBsYWJlbHM6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgICAgIGxhc3RfdG9rZW5fb25seTogYm9vbCA9IEZhbHNlLAogICAgKSAtPiBSZWNTeXNPdXRwdXQ6CiAgICAgICAgIiIiCiAgICAgICAgRm9yd2FyZCBwYXNzIHRocm91Z2ggQ2F1c2FsIEdQVC0yLgoKICAgICAgICBBcmdzOgogICAgICAgICAgICBpbnB1dF9pZHM6IExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgaXRlbSB0b2tlbnMuCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrOiBPcHRpb25hbCBMb25nVGVuc29yIFtCLCBMXSB3aXRoIDEgZm9yIHJlYWwgaXRlbXMsIDAgZm9yIDxQQUQ+LgogICAgICAgICAgICBsYWJlbHM6IE9wdGlvbmFsIExvbmdUZW5zb3IgW0IsIExdIGNvbnRhaW5pbmcgc2hpZnRlZCBncm91bmQtdHJ1dGggaXRlbSB0b2tlbnMuCiAgICAgICAgICAgIGxhc3RfdG9rZW5fb25seTogSWYgVHJ1ZSwgcHJvamVjdHMgb25seSB0aGUgZmluYWwgYWN0aXZlIGludGVyYWN0aW9uIHRva2VuLgogICAgICAgICAgICAgICAgQnlwYXNzZXMgZnVsbCBCIHggTCB4IFYgdGVuc29yIGNyZWF0aW9uIHRvIHNsYXNoIFZSQU0gZnJvbSAxMCsgR0IgdG8gPDIwMCBNQiBkdXJpbmcgZXZhbHVhdGlvbiEKCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgUmVjU3lzT3V0cHV0IGNvbnRhaW5pbmcgbG9naXRzLCBvcHRpb25hbCBzY2FsYXIgbG9zcywgYW5kIGxhc3QgaGlkZGVuIHN0YXRlLgogICAgICAgICIiIgogICAgICAgICMgQXV0by1nZW5lcmF0ZSBhdHRlbnRpb24gbWFzayBpZiBvbWl0dGVkCiAgICAgICAgaWYgYXR0ZW50aW9uX21hc2sgaXMgTm9uZToKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSAoaW5wdXRfaWRzICE9IHNlbGYucGFkX3Rva2VuX2lkKS5sb25nKCkKCiAgICAgICAgaWYgbGFzdF90b2tlbl9vbmx5OgogICAgICAgICAgICAjIE1lbW9yeS1FZmZpY2llbnQgRXZhbHVhdGlvbiBNb2RlOiBQYXNzIHRocm91Z2ggZGVjb2RlciBiYWNrYm9uZSBvbmx5CiAgICAgICAgICAgIGJhY2tib25lX291dHB1dHMgPSBzZWxmLnRyYW5zZm9ybWVyLnRyYW5zZm9ybWVyKAogICAgICAgICAgICAgICAgaW5wdXRfaWRzPWlucHV0X2lkcywKICAgICAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrLAogICAgICAgICAgICAgICAgb3V0cHV0X2hpZGRlbl9zdGF0ZXM9RmFsc2UsCiAgICAgICAgICAgICAgICByZXR1cm5fZGljdD1UcnVlLAogICAgICAgICAgICApCiAgICAgICAgICAgIGhpZGRlbl9zdGF0ZXMgPSBiYWNrYm9uZV9vdXRwdXRzLmxhc3RfaGlkZGVuX3N0YXRlICAjIFtCLCBMLCBuX2VtYmRdCgogICAgICAgICAgICAjIEV4dHJhY3QgaGlkZGVuIHN0YXRlIGF0IHRoZSBsYXN0IGFjdGl2ZSB0b2tlbgogICAgICAgICAgICBzZXFfbGVucyA9IHRvcmNoLmNsYW1wKGF0dGVudGlvbl9tYXNrLnN1bShkaW09LTEpIC0gMSwgbWluPTApCiAgICAgICAgICAgIGJhdGNoX2lkeCA9IHRvcmNoLmFyYW5nZShpbnB1dF9pZHMuc2l6ZSgwKSwgZGV2aWNlPWlucHV0X2lkcy5kZXZpY2UpCiAgICAgICAgICAgIGxhc3RfaGlkZGVuID0gaGlkZGVuX3N0YXRlc1tiYXRjaF9pZHgsIHNlcV9sZW5zXSAgIyBbQiwgbl9lbWJkXQoKICAgICAgICAgICAgIyBQcm9qZWN0IE9OTFkgdGhlIGxhc3QgdG9rZW4gdGhyb3VnaCBsbV9oZWFkIFtCLCB2b2NhYl9zaXplXSAoVGlueSBtZW1vcnkhKQogICAgICAgICAgICBsYXN0X2xvZ2l0cyA9IHNlbGYudHJhbnNmb3JtZXIubG1faGVhZChsYXN0X2hpZGRlbikgICMgW0IsIHZvY2FiX3NpemVdCgogICAgICAgICAgICBsb3NzID0gTm9uZQogICAgICAgICAgICBpZiBsYWJlbHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBsYXN0X2xhYmVscyA9IGxhYmVsc1tiYXRjaF9pZHgsIHNlcV9sZW5zXQogICAgICAgICAgICAgICAgbG9zcyA9IHNlbGYubG9zc19mY3QobGFzdF9sb2dpdHMsIGxhc3RfbGFiZWxzKQoKICAgICAgICAgICAgcmV0dXJuIFJlY1N5c091dHB1dCgKICAgICAgICAgICAgICAgIGxvZ2l0cz1sYXN0X2xvZ2l0cy51bnNxdWVlemUoMSksICAjIFtCLCAxLCB2b2NhYl9zaXplXQogICAgICAgICAgICAgICAgbG9zcz1sb3NzLAogICAgICAgICAgICAgICAgbGFzdF9oaWRkZW5fc3RhdGU9bGFzdF9oaWRkZW4udW5zcXVlZXplKDEpLAogICAgICAgICAgICApCgogICAgICAgICMgU3RhbmRhcmQgVHJhaW5pbmcgTW9kZTogRm9yd2FyZCBhY3Jvc3MgZnVsbCBzZXF1ZW5jZSB3aW5kb3cKICAgICAgICB0cmFuc2Zvcm1lcl9vdXRwdXRzID0gc2VsZi50cmFuc2Zvcm1lcigKICAgICAgICAgICAgaW5wdXRfaWRzPWlucHV0X2lkcywKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2ssCiAgICAgICAgICAgIG91dHB1dF9oaWRkZW5fc3RhdGVzPVRydWUsCiAgICAgICAgICAgIHJldHVybl9kaWN0PVRydWUsCiAgICAgICAgKQoKICAgICAgICBsb2dpdHMgPSB0cmFuc2Zvcm1lcl9vdXRwdXRzLmxvZ2l0cyAgIyBbQiwgTCwgdm9jYWJfc2l6ZV0KICAgICAgICBsYXN0X2hpZGRlbl9zdGF0ZSA9IHRyYW5zZm9ybWVyX291dHB1dHMuaGlkZGVuX3N0YXRlc1stMV0gICMgW0IsIEwsIG5fZW1iZF0KCiAgICAgICAgbG9zcyA9IE5vbmUKICAgICAgICBpZiBsYWJlbHMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICMgRmxhdHRlbiBiYXRjaCBhbmQgc2VxdWVuY2UgZGltZW5zaW9ucyBmb3IgY3Jvc3MtZW50cm9weQogICAgICAgICAgICAjIFBhZGRlZCBwb3NpdGlvbnMgaW4gbGFiZWxzICh2YWx1ZSAwKSBhcmUgaWdub3JlZCBpbiBsb3NzIGNvbXB1dGF0aW9uCiAgICAgICAgICAgIGxvc3MgPSBzZWxmLmxvc3NfZmN0KAogICAgICAgICAgICAgICAgbG9naXRzLnZpZXcoLTEsIGxvZ2l0cy5zaXplKC0xKSksCiAgICAgICAgICAgICAgICBsYWJlbHMudmlldygtMSksCiAgICAgICAgICAgICkKCiAgICAgICAgcmV0dXJuIFJlY1N5c091dHB1dCgKICAgICAgICAgICAgbG9naXRzPWxvZ2l0cywKICAgICAgICAgICAgbG9zcz1sb3NzLAogICAgICAgICAgICBsYXN0X2hpZGRlbl9zdGF0ZT1sYXN0X2hpZGRlbl9zdGF0ZSwKICAgICAgICApCgogICAgZGVmIGdldF9sYXN0X2FjdGlvbl9zdGF0ZXMoCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9pZHM6IHRvcmNoLlRlbnNvciwKICAgICAgICBhdHRlbnRpb25fbWFzazogT3B0aW9uYWxbdG9yY2guVGVuc29yXSA9IE5vbmUsCiAgICApIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiIKICAgICAgICBFeHRyYWN0cyB0aGUgY29udGV4dHVhbCByZXByZXNlbnRhdGlvbiB2ZWN0b3IgaF90IGF0IHRoZSBsYXN0IHZhbGlkIGludGVyYWN0aW9uIHRva2VuLgogICAgICAgIAogICAgICAgIEVzc2VudGlhbCBmb3IgUGhhc2UgMiAoUkwgRmluZS1UdW5pbmcpOiBUaGlzIHZlY3RvciBzZXJ2ZXMgYXMgdGhlIE1EUCBTdGF0ZSBzX3QKICAgICAgICBmb3IgdGhlIFJMIHBvbGljeSBhY3Rvci9jcml0aWMgbmV0d29ya3MuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGlucHV0X2lkczogTG9uZ1RlbnNvciBbQiwgTF0KICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s6IE9wdGlvbmFsIExvbmdUZW5zb3IgW0IsIExdCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIEZsb2F0VGVuc29yIFtCLCBuX2VtYmRdIHN0YXRlIHJlcHJlc2VudGF0aW9ucy4KICAgICAgICAiIiIKICAgICAgICBpZiBhdHRlbnRpb25fbWFzayBpcyBOb25lOgogICAgICAgICAgICBhdHRlbnRpb25fbWFzayA9IChpbnB1dF9pZHMgIT0gc2VsZi5wYWRfdG9rZW5faWQpLmxvbmcoKQoKICAgICAgICBvdXQgPSBzZWxmLmZvcndhcmQoaW5wdXRfaWRzPWlucHV0X2lkcywgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2spCiAgICAgICAgaGlkZGVuID0gb3V0Lmxhc3RfaGlkZGVuX3N0YXRlICAjIFtCLCBMLCBuX2VtYmRdCgogICAgICAgICMgRmluZCB0aGUgaW5kZXggb2YgdGhlIGxhc3QgYWN0aXZlIHRva2VuIGluIGVhY2ggc2VxdWVuY2UKICAgICAgICAjIFNlcXVlbmNlIGxlbmd0aHM6IHN1bSBhY3Jvc3MgYXR0ZW50aW9uX21hc2sKICAgICAgICBzZXFfbGVucyA9IGF0dGVudGlvbl9tYXNrLnN1bShkaW09LTEpIC0gMSAgIyAwLWluZGV4ZWQgW0JdCiAgICAgICAgc2VxX2xlbnMgPSB0b3JjaC5jbGFtcChzZXFfbGVucywgbWluPTApCgogICAgICAgIGJhdGNoX2luZGljZXMgPSB0b3JjaC5hcmFuZ2UoaW5wdXRfaWRzLnNpemUoMCksIGRldmljZT1pbnB1dF9pZHMuZGV2aWNlKQogICAgICAgIGxhc3Rfc3RhdGVzID0gaGlkZGVuW2JhdGNoX2luZGljZXMsIHNlcV9sZW5zXSAgIyBbQiwgbl9lbWJkXQogICAgICAgIHJldHVybiBsYXN0X3N0YXRlcwoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBwcmVkaWN0X25leHRfaXRlbXMoCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9pZHM6IHRvcmNoLlRlbnNvciwKICAgICAgICBhdHRlbnRpb25fbWFzazogT3B0aW9uYWxbdG9yY2guVGVuc29yXSA9IE5vbmUsCiAgICAgICAgdG9wX2s6IGludCA9IDEwLAogICAgICAgIGZpbHRlcl9oaXN0b3J5OiBib29sID0gVHJ1ZSwKICAgICkgLT4gVHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdOgogICAgICAgICIiIgogICAgICAgIEdlbmVyYXRlcyBUb3AtSyBuZXh0IGl0ZW0gcmVjb21tZW5kYXRpb25zIGZvciB0aGUgYWN0aXZlIHVzZXIgc2VxdWVuY2UuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGlucHV0X2lkczogTG9uZ1RlbnNvciBbQiwgTF0KICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s6IE9wdGlvbmFsIExvbmdUZW5zb3IgW0IsIExdCiAgICAgICAgICAgIHRvcF9rOiBOdW1iZXIgb2YgcmVjb21tZW5kYXRpb25zIHRvIHJldHVybiBwZXIgc2VxdWVuY2UuCiAgICAgICAgICAgIGZpbHRlcl9oaXN0b3J5OiBJZiBUcnVlLCBtYXNrcyBwcmV2aW91c2x5IGludGVyYWN0ZWQgaXRlbXMgc28gdGhleSBhcmUgbm90IHJlLXJlY29tbWVuZGVkLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBUdXBsZSBvZiAodG9wa19zY29yZXMgW0IsIHRvcF9rXSwgdG9wa19pbmRpY2VzIFtCLCB0b3Bfa10pLgogICAgICAgICIiIgogICAgICAgIHNlbGYuZXZhbCgpCiAgICAgICAgaWYgYXR0ZW50aW9uX21hc2sgaXMgTm9uZToKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSAoaW5wdXRfaWRzICE9IHNlbGYucGFkX3Rva2VuX2lkKS5sb25nKCkKCiAgICAgICAgb3V0ID0gc2VsZi5mb3J3YXJkKGlucHV0X2lkcz1pbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrKQogICAgICAgIGxvZ2l0cyA9IG91dC5sb2dpdHMgICMgW0IsIEwsIHZvY2FiX3NpemVdCgogICAgICAgICMgRXh0cmFjdCBsb2dpdHMgY29ycmVzcG9uZGluZyB0byB0aGUgbGFzdCBvYnNlcnZlZCBpdGVtCiAgICAgICAgc2VxX2xlbnMgPSBhdHRlbnRpb25fbWFzay5zdW0oZGltPS0xKSAtIDEKICAgICAgICBzZXFfbGVucyA9IHRvcmNoLmNsYW1wKHNlcV9sZW5zLCBtaW49MCkKICAgICAgICBiYXRjaF9pbmRpY2VzID0gdG9yY2guYXJhbmdlKGlucHV0X2lkcy5zaXplKDApLCBkZXZpY2U9aW5wdXRfaWRzLmRldmljZSkKICAgICAgICBuZXh0X2l0ZW1fbG9naXRzID0gbG9naXRzW2JhdGNoX2luZGljZXMsIHNlcV9sZW5zXSAgIyBbQiwgdm9jYWJfc2l6ZV0KCiAgICAgICAgIyBOZXZlciByZWNvbW1lbmQgdGhlIDxQQUQ+IHRva2VuIChpbmRleCAwKQogICAgICAgIG5leHRfaXRlbV9sb2dpdHNbOiwgc2VsZi5wYWRfdG9rZW5faWRdID0gLWZsb2F0KCJpbmYiKQoKICAgICAgICBpZiBmaWx0ZXJfaGlzdG9yeToKICAgICAgICAgICAgIyBNYXNrIG91dCBpdGVtcyB0aGF0IGFwcGVhciBpbiB0aGUgdXNlcidzIGhpc3RvcmljYWwgY29udGV4dAogICAgICAgICAgICBoaXN0b3J5X21hc2sgPSB0b3JjaC56ZXJvc19saWtlKG5leHRfaXRlbV9sb2dpdHMsIGR0eXBlPXRvcmNoLmJvb2wpCiAgICAgICAgICAgIGhpc3RvcnlfbWFzay5zY2F0dGVyXygxLCBpbnB1dF9pZHMsIFRydWUpCiAgICAgICAgICAgIGhpc3RvcnlfbWFza1s6LCBzZWxmLnBhZF90b2tlbl9pZF0gPSBGYWxzZSAgIyBwcmVzZXJ2ZSBwYWQgZmxhZwogICAgICAgICAgICBuZXh0X2l0ZW1fbG9naXRzID0gbmV4dF9pdGVtX2xvZ2l0cy5tYXNrZWRfZmlsbChoaXN0b3J5X21hc2ssIC1mbG9hdCgiaW5mIikpCgogICAgICAgICMgQ29tcHV0ZSB0b3BfayBzY29yZXMgYW5kIGluZGljZXMKICAgICAgICB0b3BrX3Njb3JlcywgdG9wa19pbmRpY2VzID0gdG9yY2gudG9wayhuZXh0X2l0ZW1fbG9naXRzLCBrPXRvcF9rLCBkaW09LTEpCiAgICAgICAgcmV0dXJuIHRvcGtfc2NvcmVzLCB0b3BrX2luZGljZXMKCgpkZWYgYnVpbGRfZ3B0Ml9yZWNzeXNfbW9kZWwoCiAgICBudW1faXRlbXM6IGludCwKICAgIG1heF9sZW5ndGg6IGludCA9IDY0LAogICAgbl9lbWJkOiBpbnQgPSAxMjgsCiAgICBuX2xheWVyOiBpbnQgPSAzLAogICAgbl9oZWFkOiBpbnQgPSA0LAogICAgZHJvcG91dDogZmxvYXQgPSAwLjEsCiAgICBwYWRfdG9rZW5faWQ6IGludCA9IDAsCikgLT4gUmVjU3lzR1BUMjoKICAgICIiIkZhY3RvcnkgaGVscGVyIHRvIGJ1aWxkIGFuZCByZXR1cm4gYSBSZWNTeXNHUFQyIG1vZGVsIGluc3RhbmNlLiIiIgogICAgcmV0dXJuIFJlY1N5c0dQVDIoCiAgICAgICAgbnVtX2l0ZW1zPW51bV9pdGVtcywKICAgICAgICBtYXhfbGVuZ3RoPW1heF9sZW5ndGgsCiAgICAgICAgbl9lbWJkPW5fZW1iZCwKICAgICAgICBuX2xheWVyPW5fbGF5ZXIsCiAgICAgICAgbl9oZWFkPW5faGVhZCwKICAgICAgICBkcm9wb3V0PWRyb3BvdXQsCiAgICAgICAgcGFkX3Rva2VuX2lkPXBhZF90b2tlbl9pZCwKICAgICkK",
  "train_base.py": "...",
  "sweep.py": "IiIiCnN3ZWVwLnB5Ci0tLS0tLS0tCkF1dG9tYXRlZCwgUmVzb3VyY2UtRWZmaWNpZW50IEh5cGVycGFyYW1ldGVyIFN3ZWVwIGZvciBDYXVzYWwgR1BULTIgUmVjb21tZW5kZXIuCgpGZWF0dXJlczoKMS4gT25lLVRpbWUgRGF0YSBTdWJzYW1wbGluZyAmIFByZXByb2Nlc3Npbmc6IEN1dHMgZGF0YSB0byAyMCUgdXNlcnMgb25jZSwgYXZvaWRpbmcgcmVkdW5kYW50IHByZXByb2Nlc3NpbmcuCjIuIEZhc3QgRXhwbG9yYXRpb246IFJ1bnMgTiByYW5kb20gY29uZmlndXJhdGlvbiB0cmlhbHMgZm9yIDItMyBlcG9jaHMgZWFjaC4KMy4gTGVhay1Qcm9vZiBFdmFsdWF0aW9uOiBVc2VzIGlkZW50aWNhbCB0ZW1wb3JhbCBzcGxpdCBhbmQgbWVtb3J5LWVmZmljaWVudCB0b3AtayBtZXRyaWNzLgo0LiBBdXRvLUxvZ2dpbmcgJiBQZXJzaXN0ZW5jZTogU2F2ZXMgdGFidWxhciByZXN1bHRzIHRvIENTViBhbmQgdGhlIGJlc3QgY29uZmlndXJhdGlvbiB0byBKU09OLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBEaWN0LCBMaXN0LCBPcHRpb25hbAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gub3B0aW0gaW1wb3J0IEFkYW1XCgpmcm9tIGRhdGFfdXRpbHMgaW1wb3J0IEl0ZW1Ub2tlbml6ZXIsIHByZXByb2Nlc3NfcGlwZWxpbmUKZnJvbSBkYXRhc2V0IGltcG9ydCBjcmVhdGVfZGF0YWxvYWRlcgpmcm9tIG1vZGVsIGltcG9ydCBSZWNTeXNHUFQyLCBidWlsZF9ncHQyX3JlY3N5c19tb2RlbApmcm9tIHRyYWluX2Jhc2UgaW1wb3J0IGV2YWx1YXRlLCBnZXRfZGV2aWNlLCBnZXRfbHJfc2NoZWR1bGVyLCB0cmFpbl9vbmVfZXBvY2gKCiMgUHJldmVudCBDVURBIG1lbW9yeSBmcmFnbWVudGF0aW9uCm9zLmVudmlyb25bIlBZVE9SQ0hfQ1VEQV9BTExPQ19DT05GIl0gPSAiZXhwYW5kYWJsZV9zZWdtZW50czpUcnVlIgoKCiMgSHlwZXJwYXJhbWV0ZXIgU2VhcmNoIFNwYWNlClNFQVJDSF9TUEFDRSA9IHsKICAgICJuX2VtYmQiOiBbMTI4LCAyNTYsIDM4NF0sCiAgICAibl9sYXllciI6IFszLCA0LCA2XSwKICAgICJuX2hlYWQiOiBbNCwgOF0sCiAgICAibHIiOiBbMWUtNCwgM2UtNCwgNWUtNCwgMWUtM10sCiAgICAid2FybXVwX3JhdGlvIjogWzAuMDMsIDAuMDUsIDAuMTBdLAogICAgIndlaWdodF9kZWNheSI6IFswLjAxLCAwLjA1LCAwLjFdLAogICAgImJhdGNoX3NpemUiOiBbMzIsIDY0XSwKICAgICJncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMiOiBbMiwgNF0sCn0KCgpkZWYgc2FtcGxlX2h5cGVycGFyYW1ldGVycyhybmc6IHJhbmRvbS5SYW5kb20pIC0+IERpY3Rbc3RyLCBBbnldOgogICAgIiIiU2FtcGxlcyBhIHZhbGlkIGh5cGVycGFyYW1ldGVyIGNvbmZpZ3VyYXRpb24gZ3VhcmFudGVlaW5nIG5fZW1iZCAlIG5faGVhZCA9PSAwLiIiIgogICAgd2hpbGUgVHJ1ZToKICAgICAgICBuX2VtYmQgPSBybmcuY2hvaWNlKFNFQVJDSF9TUEFDRVsibl9lbWJkIl0pCiAgICAgICAgbl9oZWFkID0gcm5nLmNob2ljZShTRUFSQ0hfU1BBQ0VbIm5faGVhZCJdKQogICAgICAgIGlmIG5fZW1iZCAlIG5faGVhZCA9PSAwOgogICAgICAgICAgICBicmVhawoKICAgIGJhdGNoX3NpemUgPSBybmcuY2hvaWNlKFNFQVJDSF9TUEFDRVsiYmF0Y2hfc2l6ZSJdKQogICAgZ3JhZF9hY2N1bSA9IHJuZy5jaG9pY2UoU0VBUkNIX1NQQUNFWyJncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMiXSkKCiAgICByZXR1cm4gewogICAgICAgICJuX2VtYmQiOiBuX2VtYmQsCiAgICAgICAgIm5fbGF5ZXIiOiBybmcuY2hvaWNlKFNFQVJDSF9TUEFDRVsibl9sYXllciJdKSwKICAgICAgICAibl9oZWFkIjogbl9oZWFkLAogICAgICAgICJsciI6IHJuZy5jaG9pY2UoU0VBUkNIX1NQQUNFWyJsciJdKSwKICAgICAgICAid2FybXVwX3JhdGlvIjogcm5nLmNob2ljZShTRUFSQ0hfU1BBQ0VbIndhcm11cF9yYXRpbyJdKSwKICAgICAgICAid2VpZ2h0X2RlY2F5Ijogcm5nLmNob2ljZShTRUFSQ0hfU1BBQ0VbIndlaWdodF9kZWNheSJdKSwKICAgICAgICAiYmF0Y2hfc2l6ZSI6IGJhdGNoX3NpemUsCiAgICAgICAgImdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcyI6IGdyYWRfYWNjdW0sCiAgICB9CgoKZGVmIHJ1bl90cmlhbCgKICAgIHRyaWFsX2lkOiBpbnQsCiAgICBjb25maWc6IERpY3Rbc3RyLCBBbnldLAogICAgdHJhaW5fZGY6IHBkLkRhdGFGcmFtZSwKICAgIHZhbF9kZjogcGQuRGF0YUZyYW1lLAogICAgdG9rZW5pemVyOiBJdGVtVG9rZW5pemVyLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UsCiAgICB1c2VfYW1wOiBib29sLAogICAgZXBvY2hzOiBpbnQsCiAgICBtYXhfbGVuZ3RoOiBpbnQsCiAgICB1c2VyX2NvbDogc3RyLAogICAgbnVtX3dvcmtlcnM6IGludCwKKSAtPiBEaWN0W3N0ciwgQW55XToKICAgICIiIkV4ZWN1dGVzIGEgc2luZ2xlIGh5cGVycGFyYW1ldGVyIGV2YWx1YXRpb24gdHJpYWwuIiIiCiAgICBwcmludChmIlxueyc9Jyo3MH0iKQogICAgcHJpbnQoZiItLS0gU3RhcnRpbmcgVHJpYWwge3RyaWFsX2lkfSAtLS0iKQogICAgcHJpbnQoCiAgICAgICAgZiJDb25maWc6IG5fZW1iZD17Y29uZmlnWyduX2VtYmQnXX0sIG5fbGF5ZXI9e2NvbmZpZ1snbl9sYXllciddfSwgIgogICAgICAgIGYibl9oZWFkPXtjb25maWdbJ25faGVhZCddfSwgbHI9e2NvbmZpZ1snbHInXX0sIHdhcm11cD17Y29uZmlnWyd3YXJtdXBfcmF0aW8nXX0sICIKICAgICAgICBmIndkPXtjb25maWdbJ3dlaWdodF9kZWNheSddfSwgYmF0Y2hfc2l6ZT17Y29uZmlnWydiYXRjaF9zaXplJ119IChhY2N1bT17Y29uZmlnWydncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMnXX0pIgogICAgKQogICAgcHJpbnQoZiJ7Jz0nKjcwfSIpCgogICAgcGluX21lbSA9IGRldmljZS50eXBlID09ICJjdWRhIgogICAgdHJhaW5fbG9hZGVyID0gY3JlYXRlX2RhdGFsb2FkZXIoCiAgICAgICAgZGY9dHJhaW5fZGYsCiAgICAgICAgbWF4X2xlbmd0aD1tYXhfbGVuZ3RoLAogICAgICAgIGJhdGNoX3NpemU9Y29uZmlnWyJiYXRjaF9zaXplIl0sCiAgICAgICAgc2h1ZmZsZT1UcnVlLAogICAgICAgIHNsaWRpbmdfd2luZG93PVRydWUsCiAgICAgICAgdXNlcl9jb2w9dXNlcl9jb2wsCiAgICAgICAgbnVtX3dvcmtlcnM9bnVtX3dvcmtlcnMsCiAgICAgICAgcGluX21lbW9yeT1waW5fbWVtLAogICAgKQoKICAgIHZhbF9sb2FkZXIgPSBjcmVhdGVfZGF0YWxvYWRlcigKICAgICAgICBkZj12YWxfZGYsCiAgICAgICAgaGlzdG9yeV9kZj10cmFpbl9kZiwKICAgICAgICBtYXhfbGVuZ3RoPW1heF9sZW5ndGgsCiAgICAgICAgYmF0Y2hfc2l6ZT1jb25maWdbImJhdGNoX3NpemUiXSwKICAgICAgICBzaHVmZmxlPUZhbHNlLAogICAgICAgIHNsaWRpbmdfd2luZG93PUZhbHNlLAogICAgICAgIHVzZXJfY29sPXVzZXJfY29sLAogICAgICAgIG51bV93b3JrZXJzPW51bV93b3JrZXJzLAogICAgICAgIHBpbl9tZW1vcnk9cGluX21lbSwKICAgICkKCiAgICBtb2RlbCA9IGJ1aWxkX2dwdDJfcmVjc3lzX21vZGVsKAogICAgICAgIG51bV9pdGVtcz10b2tlbml6ZXIubnVtX2l0ZW1zLAogICAgICAgIG1heF9sZW5ndGg9bWF4X2xlbmd0aCwKICAgICAgICBuX2VtYmQ9Y29uZmlnWyJuX2VtYmQiXSwKICAgICAgICBuX2xheWVyPWNvbmZpZ1sibl9sYXllciJdLAogICAgICAgIG5faGVhZD1jb25maWdbIm5faGVhZCJdLAogICAgKS50byhkZXZpY2UpCgogICAgb3B0aW1pemVyID0gQWRhbVcoCiAgICAgICAgbW9kZWwucGFyYW1ldGVycygpLAogICAgICAgIGxyPWNvbmZpZ1sibHIiXSwKICAgICAgICB3ZWlnaHRfZGVjYXk9Y29uZmlnWyJ3ZWlnaHRfZGVjYXkiXSwKICAgICkKCiAgICBlZmZlY3RpdmVfc3RlcHNfcGVyX2Vwb2NoID0gbWF4KAogICAgICAgIDEsCiAgICAgICAgbGVuKHRyYWluX2xvYWRlcikgLy8gY29uZmlnWyJncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMiXQogICAgICAgICsgKDEgaWYgbGVuKHRyYWluX2xvYWRlcikgJSBjb25maWdbImdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcyJdICE9IDAgZWxzZSAwKSwKICAgICkKICAgIHRvdGFsX3N0ZXBzID0gZWZmZWN0aXZlX3N0ZXBzX3Blcl9lcG9jaCAqIGVwb2NocwogICAgd2FybXVwX3N0ZXBzID0gaW50KHRvdGFsX3N0ZXBzICogY29uZmlnWyJ3YXJtdXBfcmF0aW8iXSkKCiAgICBzY2hlZHVsZXIgPSBnZXRfbHJfc2NoZWR1bGVyKAogICAgICAgIG9wdGltaXplcj1vcHRpbWl6ZXIsCiAgICAgICAgbnVtX3dhcm11cF9zdGVwcz13YXJtdXBfc3RlcHMsCiAgICAgICAgbnVtX3RyYWluaW5nX3N0ZXBzPXRvdGFsX3N0ZXBzLAogICAgKQoKICAgIHNjYWxlciA9IHRvcmNoLmFtcC5HcmFkU2NhbGVyKCJjdWRhIiwgZW5hYmxlZD11c2VfYW1wKQoKICAgIGJlc3RfdmFsX2hyID0gLTEuMAogICAgYmVzdF92YWxfbmRjZyA9IC0xLjAKICAgIGJlc3RfdmFsX2xvc3MgPSBmbG9hdCgiaW5mIikKICAgIGJlc3RfZXBvY2ggPSAwCiAgICB0cmlhbF9zdGFydF90aW1lID0gdGltZS50aW1lKCkKCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoMSwgZXBvY2hzICsgMSk6CiAgICAgICAgZXBfdDAgPSB0aW1lLnRpbWUoKQogICAgICAgIHRyYWluX2xvc3MgPSB0cmFpbl9vbmVfZXBvY2goCiAgICAgICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgICAgICBkYXRhbG9hZGVyPXRyYWluX2xvYWRlciwKICAgICAgICAgICAgb3B0aW1pemVyPW9wdGltaXplciwKICAgICAgICAgICAgc2NoZWR1bGVyPXNjaGVkdWxlciwKICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgc2NhbGVyPXNjYWxlciwKICAgICAgICAgICAgdXNlX2FtcD11c2VfYW1wLAogICAgICAgICAgICBlcG9jaD1lcG9jaCwKICAgICAgICAgICAgZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzPWNvbmZpZ1siZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIl0sCiAgICAgICAgKQoKICAgICAgICB2YWxfbWV0cmljcyA9IGV2YWx1YXRlKAogICAgICAgICAgICBtb2RlbD1tb2RlbCwKICAgICAgICAgICAgZGF0YWxvYWRlcj12YWxfbG9hZGVyLAogICAgICAgICAgICBkZXZpY2U9ZGV2aWNlLAogICAgICAgICAgICB1c2VfYW1wPXVzZV9hbXAsCiAgICAgICAgICAgIHRvcF9rPTEwLAogICAgICAgICkKICAgICAgICBlcF9lbGFwc2VkID0gdGltZS50aW1lKCkgLSBlcF90MAoKICAgICAgICB2YWxfbG9zcyA9IHZhbF9tZXRyaWNzWyJ2YWxfbG9zcyJdCiAgICAgICAgdmFsX2hyID0gdmFsX21ldHJpY3NbInZhbF9ockAxMCJdCiAgICAgICAgdmFsX25kY2cgPSB2YWxfbWV0cmljc1sidmFsX25kY2dAMTAiXQoKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiJUcmlhbCB7dHJpYWxfaWR9IHwgRXBvY2gge2Vwb2NoOjAyZH0ve2Vwb2NoczowMmR9IFt7ZXBfZWxhcHNlZDouMWZ9c10gfCAiCiAgICAgICAgICAgIGYiVHJhaW4gTG9zczoge3RyYWluX2xvc3M6LjRmfSB8IFZhbCBMb3NzOiB7dmFsX2xvc3M6LjRmfSB8ICIKICAgICAgICAgICAgZiJIUkAxMDoge3ZhbF9ocjouNGZ9IHwgTkRDR0AxMDoge3ZhbF9uZGNnOi40Zn0iCiAgICAgICAgKQoKICAgICAgICBpZiB2YWxfaHIgPiBiZXN0X3ZhbF9ocjoKICAgICAgICAgICAgYmVzdF92YWxfaHIgPSB2YWxfaHIKICAgICAgICAgICAgYmVzdF92YWxfbmRjZyA9IHZhbF9uZGNnCiAgICAgICAgICAgIGJlc3RfdmFsX2xvc3MgPSB2YWxfbG9zcwogICAgICAgICAgICBiZXN0X2Vwb2NoID0gZXBvY2gKCiAgICB0cmlhbF9kdXJhdGlvbiA9IHRpbWUudGltZSgpIC0gdHJpYWxfc3RhcnRfdGltZQoKICAgICMgQ2xlYW51cCBHUFUgbWVtb3J5CiAgICBkZWwgbW9kZWwsIG9wdGltaXplciwgc2NoZWR1bGVyLCBzY2FsZXIsIHRyYWluX2xvYWRlciwgdmFsX2xvYWRlcgogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQoKICAgIHJlc3VsdCA9IHsKICAgICAgICAidHJpYWxfaWQiOiB0cmlhbF9pZCwKICAgICAgICAiYmVzdF9lcG9jaCI6IGJlc3RfZXBvY2gsCiAgICAgICAgImJlc3RfdmFsX2hyMTAiOiByb3VuZChiZXN0X3ZhbF9ociwgNCksCiAgICAgICAgImJlc3RfdmFsX25kY2cxMCI6IHJvdW5kKGJlc3RfdmFsX25kY2csIDQpLAogICAgICAgICJiZXN0X3ZhbF9sb3NzIjogcm91bmQoYmVzdF92YWxfbG9zcywgNCksCiAgICAgICAgInRyaWFsX2R1cmF0aW9uX3NlYyI6IHJvdW5kKHRyaWFsX2R1cmF0aW9uLCAxKSwKICAgICAgICAqKmNvbmZpZywKICAgIH0KICAgIHJldHVybiByZXN1bHQKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBpc19rYWdnbGUgPSBvcy5wYXRoLmV4aXN0cygiL2thZ2dsZSIpIG9yICJLQUdHTEVfS0VSTkVMX1JVTl9UWVBFIiBpbiBvcy5lbnZpcm9uCiAgICBkZWZhdWx0X291dHB1dCA9ICIva2FnZ2xlL3dvcmtpbmcvc3dlZXBfcmVzdWx0cyIgaWYgaXNfa2FnZ2xlIGVsc2UgInN3ZWVwX3Jlc3VsdHMiCiAgICBkZWZhdWx0X3Byb2Nlc3NlZCA9ICIva2FnZ2xlL3dvcmtpbmcvZGF0YS9wcm9jZXNzZWRfc3dlZXAiIGlmIGlzX2thZ2dsZSBlbHNlICJkYXRhL3Byb2Nlc3NlZF9zd2VlcCIKCiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0iSHlwZXJwYXJhbWV0ZXIgU3dlZXAgZm9yIFJlY1N5cyBHUFQtMiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGFfcGF0aCIsIHR5cGU9c3RyLCBkZWZhdWx0PSJkYXRhL3N5bnRoZXRpY195YW1iZGEucGFycXVldCIsIGhlbHA9IlBhdGggdG8gcmF3IGludGVyYWN0aW9ucyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb2Nlc3NlZF9kaXIiLCB0eXBlPXN0ciwgZGVmYXVsdD1kZWZhdWx0X3Byb2Nlc3NlZCwgaGVscD0iRGlyZWN0b3J5IGZvciBzd2VlcCBwcm9jZXNzZWQgZGF0YSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dF9kaXIiLCB0eXBlPXN0ciwgZGVmYXVsdD1kZWZhdWx0X291dHB1dCwgaGVscD0iRGlyZWN0b3J5IHRvIHNhdmUgc3dlZXAgcmVzdWx0cyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bV90cmlhbHMiLCB0eXBlPWludCwgZGVmYXVsdD0xMiwgaGVscD0iTnVtYmVyIG9mIHJhbmRvbSBoeXBlcnBhcmFtZXRlciB0cmlhbHMgKGRlZmF1bHQ6IDEyKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXN1YnNhbXBsZV91c2VycyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4yMCwgaGVscD0iRnJhY3Rpb24gb2YgdXNlcnMgdG8gc2FtcGxlIChkZWZhdWx0OiAwLjIwID0gMjAlKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2Noc19wZXJfdHJpYWwiLCB0eXBlPWludCwgZGVmYXVsdD0zLCBoZWxwPSJUcmFpbmluZyBlcG9jaHMgcGVyIHRyaWFsIChkZWZhdWx0OiAzKSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heF9sZW5ndGgiLCB0eXBlPWludCwgZGVmYXVsdD02NCwgaGVscD0iU2VxdWVuY2UgbWF4aW11bSBsZW5ndGgiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tYXhfaXRlbXMiLCB0eXBlPWludCwgZGVmYXVsdD0xMDAwMDAsIGhlbHA9IlZvY2FidWxhcnkgY2FwIChkZWZhdWx0OiAxMDAwMDApIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZnAxNiIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9IkVuYWJsZSBtaXhlZCBwcmVjaXNpb24gdHJhaW5pbmciKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS11c2VyX2NvbCIsIHR5cGU9c3RyLCBkZWZhdWx0PSJ1aWQiLCBoZWxwPSJVc2VyIGNvbHVtbiBuYW1lIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0taXRlbV9jb2wiLCB0eXBlPXN0ciwgZGVmYXVsdD0iaXRlbV9pZCIsIGhlbHA9Ikl0ZW0gY29sdW1uIG5hbWUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10aW1lc3RhbXBfY29sIiwgdHlwZT1zdHIsIGRlZmF1bHQ9InRpbWVzdGFtcCIsIGhlbHA9IlRpbWVzdGFtcCBjb2x1bW4gbmFtZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWR1cmF0aW9uX2NvbCIsIHR5cGU9c3RyLCBkZWZhdWx0PSJsaXN0ZW5fZHVyYXRpb25fc2VjIiwgaGVscD0iRHVyYXRpb24gY29sdW1uIG5hbWUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1udW1fd29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PTIgaWYgaXNfa2FnZ2xlIGVsc2UgMCwgaGVscD0iRGF0YUxvYWRlciB3b3JrZXJzIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTQyLCBoZWxwPSJSYW5kb20gc2VlZCBmb3IgcmVwcm9kdWNpYmlsaXR5IikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgZGV2aWNlID0gZ2V0X2RldmljZSgpCiAgICBvdXRwdXRfZGlyID0gUGF0aChhcmdzLm91dHB1dF9kaXIpCiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHVzZV9hbXAgPSBhcmdzLmZwMTYgYW5kIGRldmljZS50eXBlID09ICJjdWRhIgoKICAgIHByaW50KCI9IiAqIDcwKQogICAgcHJpbnQoIkhZUEVSUEFSQU1FVEVSIFNXRUVQIEZPUiBSRUNTWVMgR1BULTIiKQogICAgcHJpbnQoZiJUcmlhbHM6IHthcmdzLm51bV90cmlhbHN9IHwgRXBvY2hzL1RyaWFsOiB7YXJncy5lcG9jaHNfcGVyX3RyaWFsfSB8IFN1YnNhbXBsZToge2FyZ3Muc3Vic2FtcGxlX3VzZXJzICogMTAwOi4wZn0lIHVzZXJzIikKICAgIHByaW50KGYiRGV2aWNlOiB7ZGV2aWNlfSB8IEZQMTY6IHt1c2VfYW1wfSB8IE91dHB1dDoge291dHB1dF9kaXJ9IikKICAgIHByaW50KCI9IiAqIDcwKQoKICAgICMgMS4gUHJlcHJvY2VzcyBkYXRhIG9uY2Ugd2l0aCB1c2VyIHN1YnNhbXBsaW5nCiAgICBwcm9jZXNzZWRfZGlyID0gUGF0aChhcmdzLnByb2Nlc3NlZF9kaXIpCiAgICB0cmFpbl9wYXJxdWV0ID0gcHJvY2Vzc2VkX2RpciAvICJ0cmFpbl9pbnRlcmFjdGlvbnMucGFycXVldCIKICAgIHZhbF9wYXJxdWV0ID0gcHJvY2Vzc2VkX2RpciAvICJ2YWxfaW50ZXJhY3Rpb25zLnBhcnF1ZXQiCiAgICB0b2tlbml6ZXJfcGF0aCA9IHByb2Nlc3NlZF9kaXIgLyAiaXRlbV90b2tlbml6ZXIuanNvbiIKCiAgICBpZiB0cmFpbl9wYXJxdWV0LmV4aXN0cygpIGFuZCB2YWxfcGFycXVldC5leGlzdHMoKSBhbmQgdG9rZW5pemVyX3BhdGguZXhpc3RzKCk6CiAgICAgICAgcHJpbnQoZiJbRGF0YV0gRm91bmQgZXhpc3RpbmcgcHJlcHJvY2Vzc2VkIHN3ZWVwIGRhdGEgaW4ge3Byb2Nlc3NlZF9kaXJ9LiBSZXVzaW5nLiIpCiAgICAgICAgdHJhaW5fZGYgPSBwZC5yZWFkX3BhcnF1ZXQodHJhaW5fcGFycXVldCkKICAgICAgICB2YWxfZGYgPSBwZC5yZWFkX3BhcnF1ZXQodmFsX3BhcnF1ZXQpCiAgICAgICAgdG9rZW5pemVyID0gSXRlbVRva2VuaXplci5sb2FkKHRva2VuaXplcl9wYXRoKQogICAgZWxzZToKICAgICAgICBwcmludChmIltEYXRhXSBQcmVwcm9jZXNzaW5nIGRhdGFzZXQgd2l0aCB7YXJncy5zdWJzYW1wbGVfdXNlcnMgKiAxMDA6LjBmfSUgdXNlciBzdWJzYW1wbGluZy4uLiIpCiAgICAgICAgdHJhaW5fZGYsIHZhbF9kZiwgXywgdG9rZW5pemVyLCBfID0gcHJlcHJvY2Vzc19waXBlbGluZSgKICAgICAgICAgICAgZGF0YV9wYXRoPWFyZ3MuZGF0YV9wYXRoLAogICAgICAgICAgICBvdXRwdXRfZGlyPXByb2Nlc3NlZF9kaXIsCiAgICAgICAgICAgIG1pbl9pdGVtX2ZyZXF1ZW5jeT01LAogICAgICAgICAgICBtYXhfaXRlbXM9YXJncy5tYXhfaXRlbXMsCiAgICAgICAgICAgIHN1YnNhbXBsZV91c2Vycz1hcmdzLnN1YnNhbXBsZV91c2VycywKICAgICAgICAgICAgc2VlZD1hcmdzLnNlZWQsCiAgICAgICAgICAgIHNwbGl0X21vZGU9Imdsb2JhbF90ZW1wb3JhbCIsCiAgICAgICAgICAgIHVzZXJfY29sPWFyZ3MudXNlcl9jb2wsCiAgICAgICAgICAgIGl0ZW1fY29sPWFyZ3MuaXRlbV9jb2wsCiAgICAgICAgICAgIHRpbWVzdGFtcF9jb2w9YXJncy50aW1lc3RhbXBfY29sLAogICAgICAgICAgICBkdXJhdGlvbl9jb2w9YXJncy5kdXJhdGlvbl9jb2wsCiAgICAgICAgKQoKICAgIHJlc3VsdHNfY3N2ID0gb3V0cHV0X2RpciAvICJzd2VlcF9yZXN1bHRzLmNzdiIKICAgIGV4aXN0aW5nX3RyaWFscyA9IHt9CiAgICBpZiByZXN1bHRzX2Nzdi5leGlzdHMoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHByZXZfZGYgPSBwZC5yZWFkX2NzdihyZXN1bHRzX2NzdikKICAgICAgICAgICAgZm9yIF8sIHJvdyBpbiBwcmV2X2RmLml0ZXJyb3dzKCk6CiAgICAgICAgICAgICAgICB0aWQgPSBpbnQocm93WyJ0cmlhbF9pZCJdKQogICAgICAgICAgICAgICAgZXhpc3RpbmdfdHJpYWxzW3RpZF0gPSByb3cudG9fZGljdCgpCiAgICAgICAgICAgIHByaW50KGYiW1Jlc3VtZV0gRm91bmQgZXhpc3Rpbmcgc3dlZXAgcmVzdWx0cyB3aXRoIHtsZW4oZXhpc3RpbmdfdHJpYWxzKX0gY29tcGxldGVkIHRyaWFscyBpbjoge3Jlc3VsdHNfY3N2fSIpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBwcmludChmIltSZXN1bWVdIENvdWxkIG5vdCByZWFkIGV4aXN0aW5nIHJlc3VsdHMgQ1NWICh7ZX0pLiBTdGFydGluZyBmcmVzaC4iKQoKICAgICMgMi4gUnVuIFN3ZWVwIFRyaWFscwogICAgcm5nID0gcmFuZG9tLlJhbmRvbShhcmdzLnNlZWQpCiAgICByZXN1bHRzOiBMaXN0W0RpY3Rbc3RyLCBBbnldXSA9IFtdCgogICAgZm9yIHRyaWFsX2lkeCBpbiByYW5nZSgxLCBhcmdzLm51bV90cmlhbHMgKyAxKToKICAgICAgICBjb25maWcgPSBzYW1wbGVfaHlwZXJwYXJhbWV0ZXJzKHJuZykKICAgICAgICBpZiB0cmlhbF9pZHggaW4gZXhpc3RpbmdfdHJpYWxzOgogICAgICAgICAgICBwcmludChmIlxuW1Jlc3VtZV0gVHJpYWwge3RyaWFsX2lkeH0gYWxyZWFkeSBmaW5pc2hlZCAoYmVzdCBIUkAxMDoge2V4aXN0aW5nX3RyaWFsc1t0cmlhbF9pZHhdLmdldCgnYmVzdF92YWxfaHIxMCcpfSkuIFNraXBwaW5nLiIpCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKGV4aXN0aW5nX3RyaWFsc1t0cmlhbF9pZHhdKQogICAgICAgICAgICBjb250aW51ZQoKICAgICAgICB0cmlhbF9yZXN1bHQgPSBydW5fdHJpYWwoCiAgICAgICAgICAgIHRyaWFsX2lkPXRyaWFsX2lkeCwKICAgICAgICAgICAgY29uZmlnPWNvbmZpZywKICAgICAgICAgICAgdHJhaW5fZGY9dHJhaW5fZGYsCiAgICAgICAgICAgIHZhbF9kZj12YWxfZGYsCiAgICAgICAgICAgIHRva2VuaXplcj10b2tlbml6ZXIsCiAgICAgICAgICAgIGRldmljZT1kZXZpY2UsCiAgICAgICAgICAgIHVzZV9hbXA9dXNlX2FtcCwKICAgICAgICAgICAgZXBvY2hzPWFyZ3MuZXBvY2hzX3Blcl90cmlhbCwKICAgICAgICAgICAgbWF4X2xlbmd0aD1hcmdzLm1heF9sZW5ndGgsCiAgICAgICAgICAgIHVzZXJfY29sPWFyZ3MudXNlcl9jb2wsCiAgICAgICAgICAgIG51bV93b3JrZXJzPWFyZ3MubnVtX3dvcmtlcnMsCiAgICAgICAgKQogICAgICAgIHJlc3VsdHMuYXBwZW5kKHRyaWFsX3Jlc3VsdCkKCiAgICAgICAgIyBJbmNyZW1lbnRhbGx5IHNhdmUgcmVzdWx0cyBDU1YgYWZ0ZXIgZXZlcnkgdHJpYWwKICAgICAgICByZXN1bHRzX2RmID0gcGQuRGF0YUZyYW1lKHJlc3VsdHMpCiAgICAgICAgcmVzdWx0c19kZi5zb3J0X3ZhbHVlcyhieT0iYmVzdF92YWxfaHIxMCIsIGFzY2VuZGluZz1GYWxzZSwgaW5wbGFjZT1UcnVlKQogICAgICAgIHJlc3VsdHNfZGYudG9fY3N2KHJlc3VsdHNfY3N2LCBpbmRleD1GYWxzZSkKCiAgICAjIDMuIEZpbmFsIFN1bW1hcnkgJiBCZXN0IENvbmZpZyBQZXJzaXN0ZW5jZQogICAgcmVzdWx0c19kZiA9IHBkLkRhdGFGcmFtZShyZXN1bHRzKQogICAgcmVzdWx0c19kZi5zb3J0X3ZhbHVlcyhieT0iYmVzdF92YWxfaHIxMCIsIGFzY2VuZGluZz1GYWxzZSwgaW5wbGFjZT1UcnVlKQogICAgYmVzdF9yb3cgPSByZXN1bHRzX2RmLmlsb2NbMF0udG9fZGljdCgpCgogICAgcHJpbnQoIlxuIiArICI9IiAqIDcwKQogICAgcHJpbnQoIlNXRUVQIENPTVBMRVRFISBUT1AgNSBDT05GSUdVUkFUSU9OUzoiKQogICAgcHJpbnQoIj0iICogNzApCiAgICBkaXNwbGF5X2NvbHMgPSBbInRyaWFsX2lkIiwgImJlc3RfdmFsX2hyMTAiLCAiYmVzdF92YWxfbmRjZzEwIiwgImJlc3RfdmFsX2xvc3MiLCAibl9lbWJkIiwgIm5fbGF5ZXIiLCAibl9oZWFkIiwgImxyIiwgIndhcm11cF9yYXRpbyIsICJ3ZWlnaHRfZGVjYXkiXQogICAgYXZhaWxhYmxlX2NvbHMgPSBbYyBmb3IgYyBpbiBkaXNwbGF5X2NvbHMgaWYgYyBpbiByZXN1bHRzX2RmLmNvbHVtbnNdCiAgICBwcmludChyZXN1bHRzX2RmW2F2YWlsYWJsZV9jb2xzXS5oZWFkKDUpLnRvX3N0cmluZyhpbmRleD1GYWxzZSkpCgogICAgYmVzdF9jb25maWcgPSB7CiAgICAgICAgIm5fZW1iZCI6IGludChiZXN0X3Jvd1sibl9lbWJkIl0pLAogICAgICAgICJuX2xheWVyIjogaW50KGJlc3Rfcm93WyJuX2xheWVyIl0pLAogICAgICAgICJuX2hlYWQiOiBpbnQoYmVzdF9yb3dbIm5faGVhZCJdKSwKICAgICAgICAibHIiOiBmbG9hdChiZXN0X3Jvd1sibHIiXSksCiAgICAgICAgIndhcm11cF9yYXRpbyI6IGZsb2F0KGJlc3Rfcm93WyJ3YXJtdXBfcmF0aW8iXSksCiAgICAgICAgIndlaWdodF9kZWNheSI6IGZsb2F0KGJlc3Rfcm93WyJ3ZWlnaHRfZGVjYXkiXSksCiAgICAgICAgImJhdGNoX3NpemUiOiBpbnQoYmVzdF9yb3dbImJhdGNoX3NpemUiXSksCiAgICAgICAgImdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcyI6IGludChiZXN0X3Jvd1siZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzIl0pLAogICAgICAgICJtYXhfbGVuZ3RoIjogYXJncy5tYXhfbGVuZ3RoLAogICAgICAgICJtYXhfaXRlbXMiOiBhcmdzLm1heF9pdGVtcywKICAgICAgICAiYmVzdF92YWxfaHIxMCI6IGZsb2F0KGJlc3Rfcm93WyJiZXN0X3ZhbF9ocjEwIl0pLAogICAgICAgICJiZXN0X3ZhbF9uZGNnMTAiOiBmbG9hdChiZXN0X3Jvd1siYmVzdF92YWxfbmRjZzEwIl0pLAogICAgfQoKICAgIGJlc3RfY29uZmlnX3BhdGggPSBvdXRwdXRfZGlyIC8gImJlc3RfY29uZmlnLmpzb24iCiAgICB3aXRoIG9wZW4oYmVzdF9jb25maWdfcGF0aCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChiZXN0X2NvbmZpZywgZiwgaW5kZW50PTIpCgogICAgcHJpbnQoZiJcbi0tPiBCZXN0IGNvbmZpZyBzYXZlZCB0bzoge2Jlc3RfY29uZmlnX3BhdGh9IikKICAgIHByaW50KGYiLS0+IEZ1bGwgc3dlZXAgcmVzdWx0cyBzYXZlZCB0bzoge291dHB1dF9kaXIgLyAnc3dlZXBfcmVzdWx0cy5jc3YnfSIpCiAgICBwcmludCgiPSIgKiA3MCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
  "evaluate_test.py": "IiIiCmV2YWx1YXRlX3Rlc3QucHkKLS0tLS0tLS0tLS0tLS0tLQpGdWxsLUNhdGFsb2cgVGVzdC1TZXQgRXZhbHVhdGlvbiBhbmQgUXVhbGl0YXRpdmUgTWFudWFsIEluc3BlY3Rpb24KZm9yIENhdXNhbCBHUFQtMiBOZXh0LUl0ZW0gUmVjb21tZW5kZXIuCgpGZWF0dXJlczoKMS4gU3RyaWN0IEhlbGQtT3V0IEV2YWx1YXRpb246IEV2YWx1YXRlcyBvbiB0ZXN0X2ludGVyYWN0aW9ucy5wYXJxdWV0IHdpdGggdHJhaW4rdmFsIGhpc3RvcmljYWwgY29udGV4dC4KMi4gTXVsdGktQ3V0b2ZmIE1ldHJpY3M6IENvbXB1dGVzIEhSQHs1LDEwLDIwfSwgTkRDR0B7NSwxMCwyMH0sIE1SUiwgVGVzdCBMb3NzLCBhbmQgUGVycGxleGl0eSBvdmVyIHRoZSBmdWxsIGNhdGFsb2cuCjMuIFF1YWxpdGF0aXZlIE1hbnVhbCBJbnNwZWN0aW9uOiBQcmludHMgdXNlciB0cmFqZWN0b3JpZXMsIGdyb3VuZC10cnV0aCB0YXJnZXRzLCBhbmQgbW9kZWwgVG9wLTEwIHByZWRpY3Rpb25zLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBMaXN0LCBPcHRpb25hbAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKZnJvbSB0cWRtIGltcG9ydCB0cWRtCgpmcm9tIGRhdGFfdXRpbHMgaW1wb3J0IEl0ZW1Ub2tlbml6ZXIKZnJvbSBkYXRhc2V0IGltcG9ydCBTZXF1ZW50aWFsUmVjRGF0YXNldCwgY3JlYXRlX2RhdGFsb2FkZXIKZnJvbSBtb2RlbCBpbXBvcnQgUmVjU3lzR1BUMiwgYnVpbGRfZ3B0Ml9yZWNzeXNfbW9kZWwKZnJvbSB0cmFpbl9iYXNlIGltcG9ydCBnZXRfZGV2aWNlCgojIFByZXZlbnQgQ1VEQSBtZW1vcnkgZnJhZ21lbnRhdGlvbgpvcy5lbnZpcm9uWyJQWVRPUkNIX0NVREFfQUxMT0NfQ09ORiJdID0gImV4cGFuZGFibGVfc2VnbWVudHM6VHJ1ZSIKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBldmFsdWF0ZV9mdWxsX3Rlc3RfbWV0cmljcygKICAgIG1vZGVsOiBSZWNTeXNHUFQyLAogICAgZGF0YWxvYWRlcjogRGF0YUxvYWRlciwKICAgIGRldmljZTogdG9yY2guZGV2aWNlLAogICAgdXNlX2FtcDogYm9vbCA9IEZhbHNlLAogICAgY3V0b2ZmczogTGlzdFtpbnRdID0gWzUsIDEwLCAyMF0sCiAgICBtaW5fZW5nYWdlZF9zZWM6IGZsb2F0ID0gMzAuMCwKKSAtPiBEaWN0W3N0ciwgQW55XToKICAgICIiIgogICAgQ29tcHV0ZXMgRHVhbC1UcmFjayByYW5raW5nIG1ldHJpY3Mgb3ZlciB0aGUgZnVsbCBpdGVtIGNhdGFsb2c6CiAgICAxLiBSYXcgQmVuY2htYXJrOiBBbGwgaGVsZC1vdXQgdGFyZ2V0cyAoc3RhbmRhcmQgYmFzZWxpbmUgY29tcGFyYWJpbGl0eSkuCiAgICAyLiBFbmdhZ2VkIEJlbmNobWFyazogT25seSB0YXJnZXRzIHdoZXJlIHVzZXIgbGlzdGVuZWQgPj0gbWluX2VuZ2FnZWRfc2VjIChzYXRpc2ZpZWQgbGlzdGVucykuCiAgICAzLiBEdXJhdGlvbi1XZWlnaHRlZCBORENHOiBSZWxldmFuY2Ugc2NhbGVkIGJ5IGxpc3RlbiBkdXJhdGlvbiBjb21wbGV0aW9uLgogICAgIiIiCiAgICBtb2RlbC5ldmFsKCkKICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKCiAgICBtYXhfayA9IG1heChjdXRvZmZzKQogICAgdG90YWxfbG9zcyA9IDAuMAogICAgbnVtX2JhdGNoZXMgPSAwCgogICAgIyBSYXcgY291bnRlcnMKICAgIHRvdGFsX3JhdyA9IDAKICAgIHJhd19oaXRzID0ge2s6IDAuMCBmb3IgayBpbiBjdXRvZmZzfQogICAgcmF3X25kY2cgPSB7azogMC4wIGZvciBrIGluIGN1dG9mZnN9CiAgICByYXdfbXJyID0gMC4wCgogICAgIyBFbmdhZ2VkIGNvdW50ZXJzIChkdXJhdGlvbiA+PSBtaW5fZW5nYWdlZF9zZWMpCiAgICB0b3RhbF9lbmdhZ2VkID0gMAogICAgZW5nYWdlZF9oaXRzID0ge2s6IDAuMCBmb3IgayBpbiBjdXRvZmZzfQogICAgZW5nYWdlZF9uZGNnID0ge2s6IDAuMCBmb3IgayBpbiBjdXRvZmZzfQogICAgZW5nYWdlZF9tcnIgPSAwLjAKCiAgICAjIER1cmF0aW9uLXdlaWdodGVkIE5EQ0cKICAgIGRfbmRjZyA9IHtrOiAwLjAgZm9yIGsgaW4gY3V0b2Zmc30KCiAgICBwYWRfaWQgPSBtb2RlbC5wYWRfdG9rZW5faWQKCiAgICBmb3IgYmF0Y2ggaW4gdHFkbShkYXRhbG9hZGVyLCBkZXNjPSJFdmFsdWF0aW5nIER1YWwtVHJhY2sgVGVzdCBTZXQiLCBsZWF2ZT1GYWxzZSwgbWluaW50ZXJ2YWw9NS4wKToKICAgICAgICBiYXRjaCA9IGJhdGNoLnRvKGRldmljZSkKCiAgICAgICAgd2l0aCB0b3JjaC5hbXAuYXV0b2Nhc3QoZGV2aWNlX3R5cGU9ImN1ZGEiLCBlbmFibGVkPXVzZV9hbXApOgogICAgICAgICAgICBvdXRwdXQgPSBtb2RlbCgKICAgICAgICAgICAgICAgIGlucHV0X2lkcz1iYXRjaC5pbnB1dF9pZHMsCiAgICAgICAgICAgICAgICBhdHRlbnRpb25fbWFzaz1iYXRjaC5hdHRlbnRpb25fbWFzaywKICAgICAgICAgICAgICAgIGxhYmVscz1iYXRjaC5sYWJlbHMsCiAgICAgICAgICAgICAgICBsYXN0X3Rva2VuX29ubHk9VHJ1ZSwKICAgICAgICAgICAgKQoKICAgICAgICB0b3RhbF9sb3NzICs9IG91dHB1dC5sb3NzLml0ZW0oKQogICAgICAgIG51bV9iYXRjaGVzICs9IDEKCiAgICAgICAgIyBFeHRyYWN0IHRhcmdldCBpdGVtcyBhbmQgZHVyYXRpb25zIGF0IGxhc3QgdmFsaWQgcG9zaXRpb24KICAgICAgICBzZXFfbGVucyA9IHRvcmNoLmNsYW1wKGJhdGNoLmF0dGVudGlvbl9tYXNrLnN1bShkaW09LTEpIC0gMSwgbWluPTApCiAgICAgICAgYmF0Y2hfaWR4ID0gdG9yY2guYXJhbmdlKGJhdGNoLmxhYmVscy5zaXplKDApLCBkZXZpY2U9ZGV2aWNlKQogICAgICAgIHRhcmdldHMgPSBiYXRjaC5sYWJlbHNbYmF0Y2hfaWR4LCBzZXFfbGVuc10KICAgICAgICBkdXJhdGlvbnMgPSBiYXRjaC5kdXJhdGlvbnNbYmF0Y2hfaWR4LCBzZXFfbGVuc10KCiAgICAgICAgbG9naXRzID0gb3V0cHV0LmxvZ2l0cwogICAgICAgIGlmIGxvZ2l0cy5kaW0oKSA9PSAzOgogICAgICAgICAgICBsb2dpdHMgPSBsb2dpdHNbOiwgMCwgOl0KICAgICAgICBsb2dpdHNbOiwgcGFkX2lkXSA9IC1mbG9hdCgiaW5mIikKCiAgICAgICAgdmFsaWRfbWFzayA9IHRhcmdldHMgIT0gcGFkX2lkCiAgICAgICAgaWYgbm90IHZhbGlkX21hc2suYW55KCk6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHRhcmdldHMgPSB0YXJnZXRzW3ZhbGlkX21hc2tdCiAgICAgICAgZHVyYXRpb25zID0gZHVyYXRpb25zW3ZhbGlkX21hc2tdCiAgICAgICAgbG9naXRzID0gbG9naXRzW3ZhbGlkX21hc2tdCiAgICAgICAgbl92YWxpZCA9IHRhcmdldHMuc2l6ZSgwKQoKICAgICAgICBfLCB0b3BrX2luZGljZXMgPSB0b3JjaC50b3BrKGxvZ2l0cywgaz1tYXhfaywgZGltPS0xKQoKICAgICAgICBmb3IgaSBpbiByYW5nZShuX3ZhbGlkKToKICAgICAgICAgICAgdGFyZ2V0ID0gdGFyZ2V0c1tpXS5pdGVtKCkKICAgICAgICAgICAgZHVyID0gbWF4KDAuMCwgZHVyYXRpb25zW2ldLml0ZW0oKSkKICAgICAgICAgICAgcHJlZHMgPSB0b3BrX2luZGljZXNbaV0udG9saXN0KCkKICAgICAgICAgICAgaXNfZW5nYWdlZCA9IChkdXIgPj0gbWluX2VuZ2FnZWRfc2VjKQoKICAgICAgICAgICAgIyBSZWxldmFuY2Ugc2NvcmUgbm9ybWFsaXplZCBieSAzLW1pbnV0ZSBzdGFuZGFyZCBjb21wbGV0aW9uCiAgICAgICAgICAgIHJlbF9zY29yZSA9IG1pbigxLjAsIGR1ciAvIDE4MC4wKQoKICAgICAgICAgICAgdG90YWxfcmF3ICs9IDEKICAgICAgICAgICAgaWYgaXNfZW5nYWdlZDoKICAgICAgICAgICAgICAgIHRvdGFsX2VuZ2FnZWQgKz0gMQoKICAgICAgICAgICAgaWYgdGFyZ2V0IGluIHByZWRzOgogICAgICAgICAgICAgICAgcmFuayA9IHByZWRzLmluZGV4KHRhcmdldCkgKyAxCiAgICAgICAgICAgICAgICByZWNpcF9yYW5rID0gMS4wIC8gcmFuawoKICAgICAgICAgICAgICAgICMgUmF3IG1ldHJpY3MKICAgICAgICAgICAgICAgIHJhd19tcnIgKz0gcmVjaXBfcmFuawogICAgICAgICAgICAgICAgZm9yIGsgaW4gY3V0b2ZmczoKICAgICAgICAgICAgICAgICAgICBpZiByYW5rIDw9IGs6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhd19oaXRzW2tdICs9IDEuMAogICAgICAgICAgICAgICAgICAgICAgICByYXdfbmRjZ1trXSArPSAxLjAgLyBtYXRoLmxvZzIocmFuayArIDEpCiAgICAgICAgICAgICAgICAgICAgICAgICMgRHVyYXRpb24td2VpZ2h0ZWQgTkRDRwogICAgICAgICAgICAgICAgICAgICAgICBkX25kY2dba10gKz0gcmVsX3Njb3JlIC8gbWF0aC5sb2cyKHJhbmsgKyAxKQoKICAgICAgICAgICAgICAgICMgRW5nYWdlZCBtZXRyaWNzCiAgICAgICAgICAgICAgICBpZiBpc19lbmdhZ2VkOgogICAgICAgICAgICAgICAgICAgIGVuZ2FnZWRfbXJyICs9IHJlY2lwX3JhbmsKICAgICAgICAgICAgICAgICAgICBmb3IgayBpbiBjdXRvZmZzOgogICAgICAgICAgICAgICAgICAgICAgICBpZiByYW5rIDw9IGs6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbmdhZ2VkX2hpdHNba10gKz0gMS4wCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbmdhZ2VkX25kY2dba10gKz0gMS4wIC8gbWF0aC5sb2cyKHJhbmsgKyAxKQoKICAgIGF2Z19sb3NzID0gdG90YWxfbG9zcyAvIG1heCgxLCBudW1fYmF0Y2hlcykKICAgIHBwbCA9IG1hdGguZXhwKG1pbihhdmdfbG9zcywgMjAuMCkpCgogICAgcmVzdWx0czogRGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInRlc3RfbG9zcyI6IHJvdW5kKGF2Z19sb3NzLCA0KSwKICAgICAgICAidGVzdF9wZXJwbGV4aXR5Ijogcm91bmQocHBsLCAyKSwKICAgICAgICAidG90YWxfcmF3X3NhbXBsZXMiOiB0b3RhbF9yYXcsCiAgICAgICAgInRvdGFsX2VuZ2FnZWRfc2FtcGxlcyI6IHRvdGFsX2VuZ2FnZWQsCiAgICAgICAgImVuZ2FnZWRfcGVyY2VudGFnZSI6IHJvdW5kKCh0b3RhbF9lbmdhZ2VkIC8gbWF4KDEsIHRvdGFsX3JhdykpICogMTAwLjAsIDEpLAogICAgICAgICJyYXdfbXJyIjogcm91bmQocmF3X21yciAvIG1heCgxLCB0b3RhbF9yYXcpLCA0KSwKICAgICAgICAiZW5nYWdlZF9tcnIiOiByb3VuZChlbmdhZ2VkX21yciAvIG1heCgxLCB0b3RhbF9lbmdhZ2VkKSwgNCksCiAgICB9CgogICAgZm9yIGsgaW4gY3V0b2ZmczoKICAgICAgICByZXN1bHRzW2YicmF3X2hyQHtrfSJdID0gcm91bmQocmF3X2hpdHNba10gLyBtYXgoMSwgdG90YWxfcmF3KSwgNCkKICAgICAgICByZXN1bHRzW2YicmF3X25kY2dAe2t9Il0gPSByb3VuZChyYXdfbmRjZ1trXSAvIG1heCgxLCB0b3RhbF9yYXcpLCA0KQogICAgICAgIHJlc3VsdHNbZiJlbmdhZ2VkX2hyQHtrfSJdID0gcm91bmQoZW5nYWdlZF9oaXRzW2tdIC8gbWF4KDEsIHRvdGFsX2VuZ2FnZWQpLCA0KQogICAgICAgIHJlc3VsdHNbZiJlbmdhZ2VkX25kY2dAe2t9Il0gPSByb3VuZChlbmdhZ2VkX25kY2dba10gLyBtYXgoMSwgdG90YWxfZW5nYWdlZCksIDQpCiAgICAgICAgcmVzdWx0c1tmImR1cmF0aW9uX25kY2dAe2t9Il0gPSByb3VuZChkX25kY2dba10gLyBtYXgoMSwgdG90YWxfcmF3KSwgNCkKCiAgICByZXR1cm4gcmVzdWx0cwoKCmRlZiBydW5fcXVhbGl0YXRpdmVfaW5zcGVjdGlvbigKICAgIG1vZGVsOiBSZWNTeXNHUFQyLAogICAgdGVzdF9kZjogcGQuRGF0YUZyYW1lLAogICAgaGlzdG9yeV9kZjogcGQuRGF0YUZyYW1lLAogICAgdG9rZW5pemVyOiBJdGVtVG9rZW5pemVyLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UsCiAgICB1c2VyX2NvbDogc3RyID0gInVpZCIsCiAgICBpdGVtX2NvbDogc3RyID0gIml0ZW1faWQiLAogICAgZHVyYXRpb25fY29sOiBzdHIgPSAibGlzdGVuX2R1cmF0aW9uX3NlYyIsCiAgICBudW1fc2FtcGxlczogaW50ID0gMTAsCiAgICB0b3BfazogaW50ID0gMTAsCikgLT4gTm9uZToKICAgICIiIlByaW50cyBxdWFsaXRhdGl2ZSBodW1hbi1yZWFkYWJsZSBpbnNwZWN0aW9uIGNhc2VzIGZyb20gcmVhbCB1c2VyIHRyYWplY3Rvcmllcy4iIiIKICAgIG1vZGVsLmV2YWwoKQogICAgcHJpbnQoIlxuIiArICI9IiAqIDgwKQogICAgcHJpbnQoIlFVQUxJVEFUSVZFIE1BTlVBTCBJTlNQRUNUSU9OOiBSRUFMIFVTRVIgVEVTVCBQUkVESUNUSU9OUyIpCiAgICBwcmludCgiPSIgKiA4MCkKCiAgICAjIFBpY2sgdXNlcnMgd2l0aCBhY3RpdmUgdGVzdCB0YXJnZXRzCiAgICB0ZXN0X3VzZXJzID0gdGVzdF9kZlt1c2VyX2NvbF0udW5pcXVlKCkKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZSg0MikKICAgIHNlbGVjdGVkX3VzZXJzID0gcm5nLmNob2ljZSh0ZXN0X3VzZXJzLCBzaXplPW1pbihudW1fc2FtcGxlcywgbGVuKHRlc3RfdXNlcnMpKSwgcmVwbGFjZT1GYWxzZSkKCiAgICBoaXRfY291bnQgPSAwCgogICAgZm9yIGlkeCwgdWlkIGluIGVudW1lcmF0ZShzZWxlY3RlZF91c2Vycywgc3RhcnQ9MSk6CiAgICAgICAgIyAxLiBHYXRoZXIgaGlzdG9yaWNhbCBzZXF1ZW5jZSB1cCB0byB0aGUgdGVzdCB0YXJnZXQKICAgICAgICB1c2VyX2hpc3RvcnkgPSBoaXN0b3J5X2RmW2hpc3RvcnlfZGZbdXNlcl9jb2xdID09IHVpZF0uc29ydF92YWx1ZXMoInRpbWVzdGFtcCIpCiAgICAgICAgdXNlcl90ZXN0ID0gdGVzdF9kZlt0ZXN0X2RmW3VzZXJfY29sXSA9PSB1aWRdLnNvcnRfdmFsdWVzKCJ0aW1lc3RhbXAiKQoKICAgICAgICBpZiBsZW4odXNlcl9oaXN0b3J5KSA9PSAwIG9yIGxlbih1c2VyX3Rlc3QpID09IDA6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGhpc3RfaXRlbXMgPSB1c2VyX2hpc3RvcnlbaXRlbV9jb2xdLnRvbGlzdCgpCiAgICAgICAgaGlzdF90b2tlbnMgPSB1c2VyX2hpc3RvcnlbIml0ZW1fdG9rZW4iXS50b2xpc3QoKQogICAgICAgIGhpc3RfZHVyYXRpb25zID0gdXNlcl9oaXN0b3J5W2R1cmF0aW9uX2NvbF0udG9saXN0KCkgaWYgZHVyYXRpb25fY29sIGluIHVzZXJfaGlzdG9yeSBlbHNlIFszMC4wXSAqIGxlbihoaXN0X2l0ZW1zKQoKICAgICAgICB0YXJnZXRfaXRlbSA9IHVzZXJfdGVzdFtpdGVtX2NvbF0uaWxvY1swXQogICAgICAgIHRhcmdldF90b2tlbiA9IHVzZXJfdGVzdFsiaXRlbV90b2tlbiJdLmlsb2NbMF0KICAgICAgICB0YXJnZXRfZHVyYXRpb24gPSB1c2VyX3Rlc3RbZHVyYXRpb25fY29sXS5pbG9jWzBdIGlmIGR1cmF0aW9uX2NvbCBpbiB1c2VyX3Rlc3QgZWxzZSAzMC4wCgogICAgICAgICMgTW9kZWwgaW5wdXQ6IGxhc3QgdXAgdG8gNjQgdG9rZW5zCiAgICAgICAgaW5wdXRfdG9rZW5zID0gaGlzdF90b2tlbnNbLTY0Ol0KICAgICAgICBpbnB1dF90ZW5zb3IgPSB0b3JjaC50ZW5zb3IoW2lucHV0X3Rva2Vuc10sIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1kZXZpY2UpCgogICAgICAgIHNjb3JlcywgdG9wa190b2tlbnMgPSBtb2RlbC5wcmVkaWN0X25leHRfaXRlbXMoaW5wdXRfdGVuc29yLCB0b3Bfaz10b3BfaykKICAgICAgICBwcmVkX3Rva2VucyA9IHRvcGtfdG9rZW5zWzBdLmNwdSgpLnRvbGlzdCgpCiAgICAgICAgcHJlZF9pdGVtcyA9IHRva2VuaXplci5pbnZlcnNlX3RyYW5zZm9ybShwcmVkX3Rva2VucykKICAgICAgICBwcm9icyA9IHRvcmNoLnNvZnRtYXgoc2NvcmVzWzBdLCBkaW09LTEpLmNwdSgpLnRvbGlzdCgpCgogICAgICAgIGlzX2hpdCA9IHRhcmdldF90b2tlbiBpbiBwcmVkX3Rva2VucwogICAgICAgIHJhbmsgPSAocHJlZF90b2tlbnMuaW5kZXgodGFyZ2V0X3Rva2VuKSArIDEpIGlmIGlzX2hpdCBlbHNlIE5vbmUKICAgICAgICBpZiBpc19oaXQ6CiAgICAgICAgICAgIGhpdF9jb3VudCArPSAxCgogICAgICAgIHByaW50KGYiXG5be2lkeDowMmR9L3tudW1fc2FtcGxlc31dIFVzZXI6IHt1aWR9IikKICAgICAgICByZWNlbnRfcHJldmlldyA9IFtmIntpdGVtfSAoe2R1cjouMGZ9cykiIGZvciBpdGVtLCBkdXIgaW4gemlwKGhpc3RfaXRlbXNbLTQ6XSwgaGlzdF9kdXJhdGlvbnNbLTQ6XSldCiAgICAgICAgcHJpbnQoZiIgIFJlY2VudCBIaXN0b3J5IChsYXN0IDQgdHJhY2tzKTogeycgLT4gJy5qb2luKHJlY2VudF9wcmV2aWV3KX0iKQogICAgICAgIHByaW50KGYiICBHcm91bmQtVHJ1dGggTmV4dCBUcmFjazogICAgICAgIHt0YXJnZXRfaXRlbX0gKGxpc3RlbmVkOiB7dGFyZ2V0X2R1cmF0aW9uOi4xZn1zKSIpCgogICAgICAgIGhpdF90YWcgPSBmIkhJVCBhdCBSYW5rICN7cmFua30gKE5EQ0c6IHsxLjAgLyBtYXRoLmxvZzIocmFuayArIDEpOi40Zn0pIiBpZiBpc19oaXQgZWxzZSAiTUlTUyAoTm90IGluIFRvcC0xMCkiCiAgICAgICAgcHJpbnQoZiIgIE1vZGVsIFJlc3VsdDogICAgICAgICAgICAgICAgICAge2hpdF90YWd9IikKICAgICAgICBwcmludCgiICBUb3AtNSBSZWNvbW1lbmRlZCBUcmFja3M6IikKICAgICAgICBmb3IgciwgKHBfaXRlbSwgcF9wcm9iKSBpbiBlbnVtZXJhdGUoemlwKHByZWRfaXRlbXNbOjVdLCBwcm9ic1s6NV0pLCBzdGFydD0xKToKICAgICAgICAgICAgbWFyayA9ICIgPC0tLSBbTUFUQ0ghXSIgaWYgcF9pdGVtID09IHN0cih0YXJnZXRfaXRlbSkgZWxzZSAiIgogICAgICAgICAgICBwcmludChmIiAgICAgI3tyOjAyZH06IHtwX2l0ZW06PDI1fSAoQ29uZmlkZW5jZToge3BfcHJvYiAqIDEwMDouMmZ9JSl7bWFya30iKQoKICAgIHByaW50KCJcbiIgKyAiLSIgKiA4MCkKICAgIHByaW50KGYiTWFudWFsIEluc3BlY3Rpb24gU2FtcGxlIEhpdCBSYXRlOiB7aGl0X2NvdW50fS97bnVtX3NhbXBsZXN9ICh7aGl0X2NvdW50IC8gbWF4KDEsIG51bV9zYW1wbGVzKSAqIDEwMDouMWZ9JSkiKQogICAgcHJpbnQoIi0iICogODApCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgaXNfa2FnZ2xlID0gb3MucGF0aC5leGlzdHMoIi9rYWdnbGUiKSBvciAiS0FHR0xFX0tFUk5FTF9SVU5fVFlQRSIgaW4gb3MuZW52aXJvbgogICAgZGVmYXVsdF9wcm9jZXNzZWQgPSAiL2thZ2dsZS93b3JraW5nL2RhdGEvcHJvY2Vzc2VkIiBpZiBpc19rYWdnbGUgZWxzZSAiZGF0YS9wcm9jZXNzZWQiCiAgICBkZWZhdWx0X2NrcHQgPSAiL2thZ2dsZS93b3JraW5nL2NoZWNrcG9pbnRzL2Jlc3RfZ3B0Ml9yZWNzeXMucHQiIGlmIGlzX2thZ2dsZSBlbHNlICJjaGVja3BvaW50cy9iZXN0X2dwdDJfcmVjc3lzLnB0IgoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJFdmFsdWF0ZSBSZWNTeXMgR1BULTIgb24gSGVsZC1PdXQgVGVzdCBTZXQiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wcm9jZXNzZWRfZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9ZGVmYXVsdF9wcm9jZXNzZWQsIGhlbHA9IkRpcmVjdG9yeSBjb250YWluaW5nIHByb2Nlc3NlZCBzcGxpdHMiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jaGVja3BvaW50X3BhdGgiLCB0eXBlPXN0ciwgZGVmYXVsdD1kZWZhdWx0X2NrcHQsIGhlbHA9IlBhdGggdG8gYmVzdF9ncHQyX3JlY3N5cy5wdCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0zMiwgaGVscD0iRXZhbHVhdGlvbiBiYXRjaCBzaXplIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbnVtX3NhbXBsZXMiLCB0eXBlPWludCwgZGVmYXVsdD0xMCwgaGVscD0iTnVtYmVyIG9mIHF1YWxpdGF0aXZlIHVzZXJzIHRvIGluc3BlY3QiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mcDE2IiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0iRW5hYmxlIEZQMTYgZXZhbHVhdGlvbiIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgIGRldmljZSA9IGdldF9kZXZpY2UoKQogICAgcHJvY2Vzc2VkX2RpciA9IFBhdGgoYXJncy5wcm9jZXNzZWRfZGlyKQogICAgY2twdF9wYXRoID0gUGF0aChhcmdzLmNoZWNrcG9pbnRfcGF0aCkKCiAgICBpZiBub3QgY2twdF9wYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQ2hlY2twb2ludCBub3QgZm91bmQgYXQ6IHtja3B0X3BhdGh9IikKCiAgICAjIDEuIExvYWQgUHJlcHJvY2Vzc2VkIERhdGEKICAgIHRyYWluX2RmID0gcGQucmVhZF9wYXJxdWV0KHByb2Nlc3NlZF9kaXIgLyAidHJhaW5faW50ZXJhY3Rpb25zLnBhcnF1ZXQiKQogICAgdmFsX2RmID0gcGQucmVhZF9wYXJxdWV0KHByb2Nlc3NlZF9kaXIgLyAidmFsX2ludGVyYWN0aW9ucy5wYXJxdWV0IikKICAgIHRlc3RfZGYgPSBwZC5yZWFkX3BhcnF1ZXQocHJvY2Vzc2VkX2RpciAvICJ0ZXN0X2ludGVyYWN0aW9ucy5wYXJxdWV0IikKICAgIHRva2VuaXplciA9IEl0ZW1Ub2tlbml6ZXIubG9hZChwcm9jZXNzZWRfZGlyIC8gIml0ZW1fdG9rZW5pemVyLmpzb24iKQoKICAgIHByaW50KGYiW0RhdGFdIExvYWRlZDogVHJhaW49e2xlbih0cmFpbl9kZik6LH0sIFZhbD17bGVuKHZhbF9kZik6LH0sIFRlc3Q9e2xlbih0ZXN0X2RmKTosfSB8IFZvY2FiPXt0b2tlbml6ZXIubnVtX2l0ZW1zOix9IikKCiAgICAjIDIuIExvYWQgQ2hlY2twb2ludCBhbmQgUmVzdG9yZSBNb2RlbAogICAgY2hlY2twb2ludCA9IHRvcmNoLmxvYWQoY2twdF9wYXRoLCBtYXBfbG9jYXRpb249ZGV2aWNlKQogICAgY2ZnID0gY2hlY2twb2ludFsibW9kZWxfY29uZmlnIl0KICAgIG1vZGVsID0gYnVpbGRfZ3B0Ml9yZWNzeXNfbW9kZWwoCiAgICAgICAgbnVtX2l0ZW1zPWNmZ1sibnVtX2l0ZW1zIl0sCiAgICAgICAgbWF4X2xlbmd0aD1jZmdbIm1heF9sZW5ndGgiXSwKICAgICAgICBuX2VtYmQ9Y2ZnWyJuX2VtYmQiXSwKICAgICAgICBuX2xheWVyPWNmZ1sibl9sYXllciJdLAogICAgICAgIG5faGVhZD1jZmdbIm5faGVhZCJdLAogICAgKS50byhkZXZpY2UpCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2hlY2twb2ludFsibW9kZWxfc3RhdGVfZGljdCJdKQogICAgbW9kZWwuZXZhbCgpCgogICAgcHJpbnQoZiJbTW9kZWxdIFJlc3RvcmVkIGNoZWNrcG9pbnQgZnJvbSBFcG9jaCB7Y2hlY2twb2ludC5nZXQoJ2Vwb2NoJywgJz8nKX0gKFZhbCBIUkAxMDoge2NoZWNrcG9pbnQuZ2V0KCd2YWxfaHIxMCcsICc/Jyl9KSIpCgogICAgIyAzLiBDcmVhdGUgRnVsbCBIaXN0b3JpY2FsIFRlc3QgRGF0YUxvYWRlcgogICAgaGlzdG9yeV9kZiA9IHBkLmNvbmNhdChbdHJhaW5fZGYsIHZhbF9kZl0sIGlnbm9yZV9pbmRleD1UcnVlKQogICAgdGVzdF9sb2FkZXIgPSBjcmVhdGVfZGF0YWxvYWRlcigKICAgICAgICBkZj10ZXN0X2RmLAogICAgICAgIGhpc3RvcnlfZGY9aGlzdG9yeV9kZiwKICAgICAgICBtYXhfbGVuZ3RoPWNmZ1sibWF4X2xlbmd0aCJdLAogICAgICAgIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9RmFsc2UsCiAgICAgICAgc2xpZGluZ193aW5kb3c9RmFsc2UsCiAgICAgICAgdXNlcl9jb2w9InVpZCIsCiAgICAgICAgbnVtX3dvcmtlcnM9MiBpZiBpc19rYWdnbGUgZWxzZSAwLAogICAgICAgIHBpbl9tZW1vcnk9KGRldmljZS50eXBlID09ICJjdWRhIiksCiAgICApCgogICAgIyA0LiBDb21wdXRlIFRlc3QgTWV0cmljcwogICAgcHJpbnQoIlxuIiArICI9IiAqIDYwKQogICAgcHJpbnQoIlJVTk5JTkcgRklOQUwgVEVTVCBTRVQgRVZBTFVBVElPTiBPVkVSIEZVTEwgQ0FUQUxPRyIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIHVzZV9hbXAgPSBhcmdzLmZwMTYgYW5kIChkZXZpY2UudHlwZSA9PSAiY3VkYSIpCiAgICBtZXRyaWNzID0gZXZhbHVhdGVfZnVsbF90ZXN0X21ldHJpY3MoCiAgICAgICAgbW9kZWw9bW9kZWwsCiAgICAgICAgZGF0YWxvYWRlcj10ZXN0X2xvYWRlciwKICAgICAgICBkZXZpY2U9ZGV2aWNlLAogICAgICAgIHVzZV9hbXA9dXNlX2FtcCwKICAgICAgICBjdXRvZmZzPVs1LCAxMCwgMjBdLAogICAgKQoKICAgIHByaW50KCJcbiIgKyAiPSIgKiA3NSkKICAgIHByaW50KCJEVUFMLVRSQUNLIEhFTEQtT1VUIFRFU1QgTUVUUklDUyAoRlVMTCBDQVRBTE9HIFJBTktJTkcpOiIpCiAgICBwcmludCgiPSIgKiA3NSkKICAgIHByaW50KGYiVG90YWwgVGVzdCBVc2Vyczoge21ldHJpY3NbJ3RvdGFsX3Jhd19zYW1wbGVzJ106LH0gfCBFbmdhZ2VkIFBsYXlzICg+PTMwcyk6IHttZXRyaWNzWyd0b3RhbF9lbmdhZ2VkX3NhbXBsZXMnXTosfSAoe21ldHJpY3NbJ2VuZ2FnZWRfcGVyY2VudGFnZSddfSUpIikKICAgIHByaW50KGYiVGVzdCBMb3NzOiB7bWV0cmljc1sndGVzdF9sb3NzJ106LjRmfSB8IFBlcnBsZXhpdHk6IHttZXRyaWNzWyd0ZXN0X3BlcnBsZXhpdHknXTouMmZ9IikKICAgIHByaW50KCItIiAqIDc1KQogICAgcHJpbnQoZiJ7J01ldHJpYyc6PDE4fSB8IHsnUmF3IEJlbmNobWFyayAoQWxsKSc6PDIyfSB8IHsnRW5nYWdlZCBQbGF5cyAoPj0zMHMpJzo8MjJ9IHwgeydMaWZ0J30iKQogICAgcHJpbnQoIi0iICogNzUpCiAgICBmb3IgayBpbiBbNSwgMTAsIDIwXToKICAgICAgICByYXdfaCA9IG1ldHJpY3NbZiJyYXdfaHJAe2t9Il0KICAgICAgICBlbmdfaCA9IG1ldHJpY3NbZiJlbmdhZ2VkX2hyQHtrfSJdCiAgICAgICAgbGlmdF92YWwgPSAoKGVuZ19oIC0gcmF3X2gpIC8gbWF4KDAuMDAwMSwgcmF3X2gpKSAqIDEwMC4wCiAgICAgICAgbGlmdF9zdHIgPSBmIit7bGlmdF92YWw6LjFmfSUiIGlmIGxpZnRfdmFsID49IDAgZWxzZSBmIntsaWZ0X3ZhbDouMWZ9JSIKICAgICAgICBwcmludChmIkhpdFJhdGVAe2s6PDEwfSB8IHtyYXdfaDo8MjIuNGZ9IHwge2VuZ19oOjwyMi40Zn0gfCB7bGlmdF9zdHJ9IikKICAgIHByaW50KCItIiAqIDc1KQogICAgZm9yIGsgaW4gWzUsIDEwLCAyMF06CiAgICAgICAgcmF3X24gPSBtZXRyaWNzW2YicmF3X25kY2dAe2t9Il0KICAgICAgICBlbmdfbiA9IG1ldHJpY3NbZiJlbmdhZ2VkX25kY2dAe2t9Il0KICAgICAgICBsaWZ0X3ZhbCA9ICgoZW5nX24gLSByYXdfbikgLyBtYXgoMC4wMDAxLCByYXdfbikpICogMTAwLjAKICAgICAgICBsaWZ0X3N0ciA9IGYiK3tsaWZ0X3ZhbDouMWZ9JSIgaWYgbGlmdF92YWwgPj0gMCBlbHNlIGYie2xpZnRfdmFsOi4xZn0lIgogICAgICAgIHByaW50KGYiTkRDR0B7azo8MTN9IHwge3Jhd19uOjwyMi40Zn0gfCB7ZW5nX246PDIyLjRmfSB8IHtsaWZ0X3N0cn0iKQogICAgcHJpbnQoIi0iICogNzUpCiAgICBwcmludChmIk1SUiAgICAgICAgICAgICAgICB8IHttZXRyaWNzWydyYXdfbXJyJ106PDIyLjRmfSB8IHttZXRyaWNzWydlbmdhZ2VkX21yciddOjwyMi40Zn0gfCIpCiAgICBwcmludChmIkR1cmF0aW9uLU5EQ0dAMTAgICB8IHttZXRyaWNzWydkdXJhdGlvbl9uZGNnQDEwJ106PDIyLjRmfSB8IChSZWxldmFuY2Ugc2NhbGVkIGJ5IGxpc3RlbiBkdXJhdGlvbikiKQogICAgcHJpbnQoIj0iICogNzUpCgogICAgIyBTYXZlIHRlc3QgbWV0cmljcwogICAgbWV0cmljc19wYXRoID0gY2twdF9wYXRoLnBhcmVudCAvICJ0ZXN0X2V2YWx1YXRpb25fbWV0cmljcy5qc29uIgogICAgd2l0aCBvcGVuKG1ldHJpY3NfcGF0aCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChtZXRyaWNzLCBmLCBpbmRlbnQ9MikKICAgIHByaW50KGYiW0V4cG9ydF0gU2F2ZWQgZHVhbC10cmFjayB0ZXN0IG1ldHJpY3MgdG86IHttZXRyaWNzX3BhdGh9IikKCiAgICAjIDUuIFF1YWxpdGF0aXZlIEluc3BlY3Rpb24KICAgIHJ1bl9xdWFsaXRhdGl2ZV9pbnNwZWN0aW9uKAogICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgIHRlc3RfZGY9dGVzdF9kZiwKICAgICAgICBoaXN0b3J5X2RmPWhpc3RvcnlfZGYsCiAgICAgICAgdG9rZW5pemVyPXRva2VuaXplciwKICAgICAgICBkZXZpY2U9ZGV2aWNlLAogICAgICAgIG51bV9zYW1wbGVzPWFyZ3MubnVtX3NhbXBsZXMsCiAgICAgICAgdG9wX2s9MTAsCiAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
  "reward_model.py": "..."
}

current_dir = os.path.abspath(os.getcwd())
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

for fname, b64_code in EMBEDDED_MODULES.items():
    dest_path = os.path.join(current_dir, fname)
    with open(dest_path, "wb") as f:
        f.write(base64.b64decode(b64_code))
    print(f"[Sync] Up-to-date module unpacked: {fname}")

print(f"\nAll modules ready in {current_dir}:")
for f in sorted(os.listdir(current_dir)):
    if f.endswith('.py'):
        print(f"  - {f}")


## 3. Install Dependencies

In [ ]:
!pip install --quiet transformers huggingface_hub pandas pyarrow tqdm wandb
print("Dependencies verified and ready!")

## 4. Optional: Weights & Biases (WandB) Setup
You can securely store your `WANDB_API_KEY` under Kaggle **Add-ons -> Secrets**.

In [ ]:
import os
USE_WANDB = False

try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key)
        print("[WandB] Successfully logged in via Kaggle Secrets!")
        USE_WANDB = True
    else:
        print("[WandB] WANDB_API_KEY secret not found. Running with local console logging.")
except Exception as e:
    print("[WandB] WandB not configured. Running with local console logging.")

## 5. Dataset Setup: Download Official Yambda or Use Mounted Input
Checks `/kaggle/input/` for attached datasets. If none is found, downloads **Yambda-50M** (`flat/50m/listens.parquet`) directly from Hugging Face.

In [ ]:
DOWNLOAD_REAL_YAMBDA = True

import os
import glob
from huggingface_hub import hf_hub_download

os.makedirs("/kaggle/working/data", exist_ok=True)
data_path = None
synthetic_flag = ""

# 1. Check if user attached a dataset via /kaggle/input/
input_files = glob.glob("/kaggle/input/**/*", recursive=True)
parquet_files = [f for f in input_files if f.endswith(".parquet") or f.endswith(".csv")]

if parquet_files:
    data_path = parquet_files[0]
    print(f"[Dataset] Found mounted dataset in /kaggle/input/: {data_path}")
elif DOWNLOAD_REAL_YAMBDA:
    print("[Dataset] Downloading official Yambda-50M from Hugging Face (~300MB)... ")
    try:
        data_path = hf_hub_download(
            repo_id="yandex/yambda",
            repo_type="dataset",
            filename="flat/50m/listens.parquet",
            local_dir="/kaggle/working/data"
        )
        print(f"[Dataset] Successfully downloaded Yambda to: {data_path}")
    except Exception as e:
        print(f"[Dataset] Download failed ({e}). Falling back to synthetic generator.")
        data_path = "/kaggle/working/data/synthetic_yambda.parquet"
        synthetic_flag = "--synthetic"
else:
    data_path = "/kaggle/working/data/synthetic_yambda.parquet"
    synthetic_flag = "--synthetic"
    print("[Dataset] Running in synthetic demo mode.")


## 6. Fast Hyperparameter Sweep (Recommended)
Instead of running 10 full epochs blindly (which takes ~5 hours), this sweep evaluates multiple configurations on a **20% user subsample** for 3 epochs each (~6 min/trial).

- Searches over: `n_embd` (128, 256, 384), `n_layer` (3, 4, 6), `n_head` (4, 8), `lr`, `warmup_ratio`, `weight_decay`
- Saves ranking table to `/kaggle/working/sweep_results/sweep_results.csv`
- Automatically saves the winning config to `/kaggle/working/sweep_results/best_config.json`

In [ ]:
# 1. Restore completed trials 1-4 so sweep.py skips directly to Trial 5
import os
import pandas as pd
from pathlib import Path

sweep_dir = Path("/kaggle/working/sweep_results")
sweep_dir.mkdir(parents=True, exist_ok=True)
csv_file = sweep_dir / "sweep_results.csv"

if not csv_file.exists():
    completed_trials = [
        {"trial_id": 1, "best_epoch": 3, "best_val_hr10": 0.1746, "best_val_ndcg10": 0.1026, "best_val_loss": 7.6193, "trial_duration_sec": 1698.0, "n_embd": 384, "n_layer": 3, "n_head": 4, "lr": 0.0003, "warmup_ratio": 0.03, "weight_decay": 0.1, "batch_size": 32, "gradient_accumulation_steps": 4},
        {"trial_id": 2, "best_epoch": 3, "best_val_hr10": 0.0504, "best_val_ndcg10": 0.0255, "best_val_loss": 9.3200, "trial_duration_sec": 912.7, "n_embd": 128, "n_layer": 3, "n_head": 4, "lr": 0.0001, "warmup_ratio": 0.03, "weight_decay": 0.01, "batch_size": 64, "gradient_accumulation_steps": 2},
        {"trial_id": 3, "best_epoch": 3, "best_val_hr10": 0.1881, "best_val_ndcg10": 0.1165, "best_val_loss": 7.6853, "trial_duration_sec": 1695.0, "n_embd": 384, "n_layer": 3, "n_head": 4, "lr": 0.001, "warmup_ratio": 0.1, "weight_decay": 0.05, "batch_size": 32, "gradient_accumulation_steps": 4},
        {"trial_id": 4, "best_epoch": 3, "best_val_hr10": 0.0983, "best_val_ndcg10": 0.0547, "best_val_loss": 8.5484, "trial_duration_sec": 953.9, "n_embd": 128, "n_layer": 4, "n_head": 4, "lr": 0.0003, "warmup_ratio": 0.03, "weight_decay": 0.05, "batch_size": 64, "gradient_accumulation_steps": 4},
    ]
    pd.DataFrame(completed_trials).to_csv(csv_file, index=False)
    print(f"[Restore] Restored trials 1-4 into {csv_file}. sweep.py will now resume directly from Trial 5!")

# 2. Run Hyperparameter Sweep (resumes at Trial 5, only runs remaining trials)
!python sweep.py \
    --data_path {data_path} \
    --processed_dir /kaggle/working/data/processed_sweep \
    --output_dir /kaggle/working/sweep_results \
    --num_trials 6 \
    --epochs_per_trial 3 \
    --subsample_users 0.20 \
    --max_items 100000 \
    --max_length 64 \
    --fp16 \
    --num_workers 2


## 7. Launch Full Training (With Best Sweep Config or Resuming Checkpoint)
This cell automatically reads the winning hyperparameters from `/kaggle/working/sweep_results/best_config.json` (if available) or defaults to the upgraded capacity architecture (`n_embd=256, n_layer=4, n_head=8, lr=5e-4, warmup_ratio=0.05`).

> **Resume Tip**: If you already have a saved checkpoint from a previous session, set `RESUME_CHECKPOINT = "/kaggle/working/checkpoints/best_gpt2_recsys.pt"` (or path from `/kaggle/input/...`) to resume weights directly!

In [ ]:
import json
import os

# Check if sweep generated a best config
sweep_cfg_path = "/kaggle/working/sweep_results/best_config.json"
if os.path.exists(sweep_cfg_path):
    with open(sweep_cfg_path, "r") as f:
        best_cfg = json.load(f)
    print(f"[Config] Loaded best config from sweep: {best_cfg}")
    n_embd = best_cfg.get("n_embd", 256)
    n_layer = best_cfg.get("n_layer", 4)
    n_head = best_cfg.get("n_head", 8)
    lr = best_cfg.get("lr", 5e-4)
    warmup = best_cfg.get("warmup_ratio", 0.05)
    wd = best_cfg.get("weight_decay", 0.01)
    bs = best_cfg.get("batch_size", 32)
    grad_accum = best_cfg.get("gradient_accumulation_steps", 4)
else:
    print("[Config] Using default upgraded capacity config")
    n_embd, n_layer, n_head = 256, 4, 8
    lr, warmup, wd = 5e-4, 0.05, 0.01
    bs, grad_accum = 32, 4

# Set path to resume weights from an existing checkpoint if desired
RESUME_CHECKPOINT = None  # e.g., "/kaggle/working/checkpoints/best_gpt2_recsys.pt"
resume_flag = f"--resume_from {RESUME_CHECKPOINT}" if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT) else ""
wandb_arg = "--use_wandb" if USE_WANDB else ""

# Launch full training with linear warmup, early stopping, and FP16
!python train_base.py \
    --data_path {data_path} \
    --processed_dir /kaggle/working/data/processed \
    --output_dir /kaggle/working/checkpoints \
    --user_col uid \
    --min_item_freq 5 \
    --max_items 100000 \
    --max_length 64 \
    --n_embd {n_embd} \
    --n_layer {n_layer} \
    --n_head {n_head} \
    --batch_size {bs} \
    --gradient_accumulation_steps {grad_accum} \
    --lr {lr} \
    --warmup_ratio {warmup} \
    --weight_decay {wd} \
    --early_stopping_patience 3 \
    --epochs 5 \
    --split_mode global_temporal \
    --fp16 \
    --num_workers 2 \
    {resume_flag} \
    {synthetic_flag} \
    {wandb_arg}


## 8. Dual-Track Model Verification & Qualitative Inspection
Runs dual-track evaluation on the held-out test set (comparing raw baseline vs engaged >=30s performance) and inspects real user trajectories.

In [ ]:
# Run full dual-track test evaluation and print 10 qualitative user recommendation samples
!python evaluate_test.py \
    --processed_dir /kaggle/working/data/processed \
    --checkpoint_path /kaggle/working/checkpoints/best_gpt2_recsys.pt \
    --batch_size 32 \
    --num_samples 10 \
    --fp16


## 9. Phase 2: Train Reward Model / Value Head (Critic)
Trains an MLP Value Head on top of the Phase 1 GPT-2 representations to estimate continuous listening duration and skip probability.

In [ ]:
!python reward_model.py \
    --processed_dir /kaggle/working/data/processed \
    --base_checkpoint /kaggle/working/checkpoints/best_gpt2_recsys.pt \
    --output_dir /kaggle/working/checkpoints \
    --batch_size 64 \
    --epochs 3 \
    --fp16


## 10. Export Trained Artifacts
Compresses Phase 1 Base Model checkpoints, Phase 2 Reward Model checkpoints, and tokenizer mappings into a `.zip` archive for one-click download from `/kaggle/working`.

In [ ]:
!zip -r /kaggle/working/recsys_checkpoints.zip /kaggle/working/checkpoints /kaggle/working/sweep_results /kaggle/working/data/processed
print("Artifacts archived successfully to: /kaggle/working/recsys_checkpoints.zip")
